# Conformer-ByteHum Final V5: train sơ bộ trên Kaggle T4

Tải notebook này lên Kaggle, gắn `datasets.zip` và checkpoint Stage 1
`coverhunter_shs100k_pretrained.pth`, bật GPU T4 và Internet, rồi chọn Run All.
Notebook giải nén ZIP vào `/kaggle/working`. Nếu Kaggle gắn dữ liệu dưới dạng thư mục
chỉ đọc, notebook sẽ sao chép riêng `dataset_split` sang `/kaggle/working` trước khi
sắp xếp lại bài hát theo query ID và kiểm tra manifest/hash.

## Trạng thái dữ liệu

`datasets.zip` đang dùng split cũ. Notebook sửa split trong bản giải nén bằng
source query index đã xác minh; file ZIP gốc không bị thay đổi. Query test được
giữ trong `test` và không được dùng cho loss hoặc chọn checkpoint.

Hai fragment trong ZIP không có `reference.wav`; notebook dùng đúng bài hát tương ứng trong `2_music_database` làm positive fallback và ghi chúng vào log/data contract.

Nguồn hiện thiếu 29 query train, 3 validation và 7 test. Notebook chỉ train trên
3.583 query train và 763 query validation hiện có. Đây là run sơ bộ; khôi phục
32 query train/validation còn thiếu trước khi dùng kết quả làm run cuối. Sáu trăm
mười lăm query holdout sạch vẫn được kiểm tra bằng manifest khóa và không tham gia
train/validation.

Nếu Kaggle có nhiều dataset/checkpoint cùng loại, đặt đúng đường dẫn tại cell cấu
hình. Dùng một `OUTPUT_ROOT` mới khi bắt đầu từ Stage 1. Resume chỉ với LAST schema
2 được tạo bởi chính notebook này.

Notebook giữ nguyên cấu hình, mô hình, loss, và scoring V5. Nó không tuyên bố đã
train trên GPU; cần xem trạng thái và logs trong `/kaggle/working/qbh_final_v5_dataset_fixed`.


In [ ]:
# Edit paths only, before Run All. Learning/scoring settings are frozen below.
INPUT_ROOT = '/kaggle/input'
OUTPUT_ROOT = '/kaggle/working/qbh_final_v5_dataset_fixed'
DATASET_OVERRIDE = None  # Exact dataset_split directory if multiple inputs contain it.
MUSIC_OVERRIDE = None   # Exact 2_music_database directory if ambiguous.
DATASET_ARCHIVE_OVERRIDE = None  # Optional ZIP/.bin containing both dataset directories.
STAGE1_OVERRIDE = None  # coverhunter_shs100k_pretrained.pth (training only).
CHECKPOINT_OVERRIDE = None  # qbh_final_v5_best.pth (evaluation only).
RESUME = True
RESUME_LAST_OVERRIDE = None  # Matching schema-2 LAST; includes the best model snapshot.
RESUME_BEST_OVERRIDE = None  # Legacy setting: BEST alone cannot resume; leave None.
EVAL_RESUME_DIR = None  # Prior final-v5 test output folder; auto-detected when unique.
KNOWN_MISSING_SOURCE_IDS = ('12777492f4c9adb9', '1b474bae1557a0a6', '1f79db7550ca5772', '21f08f46c5dec2fa', '2adad3180e590a3a', '2ea10a2837697f48', '2fab930015f8d575', '342f50db0e5ebed5', '3bab87dd00f0facb', '451b846328464d65', '462b38e882e8cd78', '49a20d60e3e08806', '4b48293d9ca92cbd', '4e73f1ae5a45f82e', '548e61c546a1c145', '59b516c09115b3cf', '606bacde410ba841', '609716be5d522942', '6c376f8dfa4c0223', '6f73450e8574c99f', '76071fd87e6868ac', '7aadb74954cbd26f', '8455a49d7c5619e3', '84819d82da591aeb', '8cb95eb3955ac409', '9287ecbd7717673c', 'a37811befd2456a1', 'a9bbacd43cab1652', 'ac0e51fd30446060', 'c1c966ef839ade42', 'c8f211bf97d38ddb', 'ce96551498969d69', 'dbe9902d72fb658b', 'e4b143c7119adf13', 'e4cdb670e61fa685', 'e7da3058a6501def', 'e8a16fee7f810f17', 'e9e41f8e936d07fd', 'f7ac30225541952c')
SOURCE_QUERY_INDEX_ZLIB_B64 = (
    '...'
    '6WTrf/2///v/AbNCEPk='
)
SOURCE_QUERY_MANIFEST_SHA256 = 'ba0614b4af0f7aca6b5454946d5553a6671aba3db2570520ed7813b89627ad4a'
SOURCE_QUERY_INDEX_SHA256 = '9072a3b2ac11dc117508df7deb046d6e0fff58640cbe9ad78b3d4f9ec93dd3ff'
SOLUTION_PHASE = 'train'
SOURCE_CODE_DIGEST = '9bcb43b730813563d21294824b1a8283522b81af3f9fc756c24bbee97a4b6055'

# Verified package metadata used to repair the legacy split after safe extraction.
VERIFIED_SPLIT_MANIFEST_ZLIB_B64 = (
    'eNq9fdlTHLmW9/v8FYob80BH0A5JqVz0vbEbGzBD0fbteSG0HFF5ycrkZlWBqyfmf5+jLJYCSkXKZH93IsZ2G+fvHC1n01n+5z8I'
    '+ccWk3n2G7lozM2C/E52y6oie009K/EPlOH/O2rqWpHDakHOmvt//D/yP/iv8N9Nb6tyhn/8x6xVZf2P7Yf/2tTXV6X1/11LTm1G'
    'VUGNptSlzz8ybw1czcpZBf7nPH5v+Idv1PPJ1Xg+mZT19RS/kb789ETVpYPp7Or1j+FP/e92x/RR1bSlOp1PS/O9tND8FsGYcpYX'
    'Ji2sYwVomYQY+52sgVnPAMv7ccDyJxa2IyhmhjFJqaQ85TrLVYji7p+EfvgVKZz1I5nzJ5I5+aOuykk5A+s3FmbkApRdkMOmJZfj'
    'chrBkUszllPFZZKp1BQmxNG7kOuZS0Q/5hLxzBz7RA4qcg6tbWrEOiv9if6iJmRBDuq2/PccyPF1BdNSTcnWN+dKU6qKdGdjHeOI'
    'tm4nEyW4S5jKWWqZgiDfH6EmcEaLnme0eF4TcXPUVNYTcNo0lrjZJ1Kquia2bNqI3aaFSgvI8lwyR20SPL+b8dazJftxJZ+YSskI'
    'TFPbKWkcGeFPQIt4B2paQgxPKc0YFUKqRFJZCB3iaSNcYKdoz52i7zD1ZzOvr3WFSxmzWZQ5riGnqXBM8yKOsRXIgWT9zu7uDn54'
    'X9UG/4781xyg7n/h8jyHnDOWZ5AIoDLEzVqU9RzkPSVn/ixcdvb29/Drn8vr8b1akFlDPgMqyq1q0ZZm+lt/dqg2OjfWZIZZpZgN'
    'svMO4EB3aeds//Rg9Nkj7Zz9uUMujs/2/4hRAy63mqaFy3LUxoYF2VmLM9QBMwZuZ/jtnYmaNFNyoshdaVUEG4IqoJrThCe5FDYo'
    'C0JIASXW85wlbIUV8LdxV03Bo+COX47VjIzGQH6oehajoaXilGnUVwnlUgoT5ukdyID5waPNj5dIu6Dms9LNK3KC/zBmswQHKwvj'
    'hKMq4Wk/xl7BBQR3z7PHVg6fRZ2+V6nFrNPzp+V0WuLvjie3Df5OVzGc4XbJIkuyHLRhSSqCnL2LGTiQPS2IpFhhDyq/fCfNHXhK'
    'Y/QQp5mwhbBoxjJXJGFuXkEEDlxPE53nb4i/aKrKK4WyJjM81vsAt2v4uFPVGi4Kq3hmcg2ohSzn5j0u1mEF5FzSU9AlbxgaQW3J'
    '6aJbMrJ12Xit3ZIzuO/+SxvjTDEOKZWa68TiZYLsPf7ehQ7crJ7igvG3zN4snPK678lKPvFKkET7jVoKKXmiE0FNSlP1LqubgQN3'
    'rKcISdK3jDYTaGpAAXUDflkjWEtcTq0U6C+qXDOm32XtNVTgzvU0ZPmzIbvj/BKNyon/PKp75VSN/9f7vjFmbZpSZIOihaTD0j2A'
    'M5zkq1SNGrC66cziQ2XBRlitOk3QMGKZTDOXZeGjtgZkMJ9i5ePbZKR0W9aK7Kn2FuoZ4im8yIeqbec3DWJ/q/2t/qEWMRJeou0n'
    'bWqcdcZY2ofL/nSsX4eeG7m6j2BQ+9f/Uvh3Xs3AjJyiYG7uazKqmvsqhmPHQRbUaZGCpUqH79n7oIFt7mnCc7HK4HTql+4ztA1M'
    'ydY9ENPMK0s0/Na54JdeVp80/Y8vWsEydwlnmbKOumwDmz2hh7O3KvhZqu7O76Dlg8oWLZ+mOzQjRCZbh20zId9VBS3+WHe4vB7e'
    'K2cx+yyUTdBJS1HnQ5awdMMC/DI1w9nVVUmO1Kz0VBzPpl6ev9ZZ07C2XC+GTVoAyuBEMAWAdml4AVaxrzaDD3gIKnJMdqY3MVc3'
    'EdqxNM9lqnTOwobc07cHFMS3Y3WHp8Ov0mHTAhpKy0CLF3iPa9UtUkwAK9eCem+Ba2rTfMMlDaK/wh7uRE5QhJc1jJv50vlS5gZF'
    'Admt8Df9xZByTGaQpdYVNhciHPvZCDfcNtbX82l3h7/Mq1KR0Wxpe3yZT2d4/XebRe/rJY1MHXNWMjyMTIXly/uQw7HXKg1TH5j2'
    'LqWty+vxjFhVm7iwauFEgmrSCeFEUYQ524AWOIc9VT9f0f1e6ipy1OIKeqCctA8/1PMEGiaMU4W16I0zyMMeeQBoqAjXq8/vq/oa'
    'NS4ejR8N/uuI7QGOy54ooCkzCdptfRl6jTicKH8FdNRYUuKRJ/exnOVpYYRgQoLjJoe+nL0CDDDW01dnSZCxb97dUnhtL8sJRO0Y'
    'Z+CoSITIuaJ5X75e4v1NB3GEesNL2u7XraW2Xxp+/sGrxJtdq3/FMCt4XkjGBS9oYq3py2w/OobyKF6B6xbUDZnfkvtyNiYLHw+5'
    'LtsKHVSo7TYpJ0Sj1o16wknR5jSmSCDB1ZCq7zJspISUV4+U/G231xMwG5cx99aoLFU8A1ukVKd5EcNrB/XB53EWZGY2VjWu5jap'
    '4ecsgqOcU2mYzI1QQuW29419wFvC/S1ndWlITGfok+6WoLugw2g2R2upOzB/RD0vOE6VLVCVMFlsiNPEERA4mVnPk5k9M36nyKn6'
    'id//6v+yQ+4eBWPeUBIhVe4oz9JCagphLygMNtxVe8IY3QPMiJ7PyPl0YcYxPoPIaFpkYChzqTYme5+hN2Afi1uwlbfHu7ZEDxFd'
    'k+tlCA+UJTvaBwx+qNl6o3O9leYShiYn47lghgobNmreQxzo0u3u7vn3s7ZqvB8yhgmQUeDtZD1DPEP1Z5JUSu4KVIUhhkJAAZEo'
    'omNLuyc7e1/Pj8+++tVq7jvvvosZ+zfCCH6UcJLnJpVWpWkafAraDDfYVVqFOTZA9lCVTMhWJ4JGUIGXV80E/vqNnH6PMV6stWi9'
    'AJM6UyLsLKzCX23Ev/IEDKbfdi9HZAtdRvSVEehkfvcb2XKoRD+Rz6qawuK3Lh4QowNcnhRcaE1zxUwaVHQd8G/kajP61RI+EMDv'
    'Gb9/wez+olY+9ew5zhG1n0VqE/8Ym2Q8cYJn77H3CHf1Am8w/9zDHO58PSAn374f/CpLPM+ZSqhTqeY6/Jr5yNITXi+efuEmKmNU'
    'q3zeEUzRem+3yTHZU2gdNM11iab8aKbaW/wJtB0o/0SzT2hX5TGPfkJSmXJuC5szkOGw3xtK+hHywbdquboS4y50Zcsa2i6euvvj'
    'O+5/lpPRHE9VjEntElNoNM+E5BIKt4HndzCHUoiq9baezzODx0ikXpDzEmYtOm236rpuqvKX0qkseg8ohHJKrbMqzGkcAcPJXJRw'
    '0xl6ZdPukRLtwuO6+zXmCOcZRUeCcw7Gch3eziDWcDIIbXZyAT/H/qIcT/yjB0yjspJoKilFp1bzVAjJwrysAl09IQ32Tv0C4BRU'
    'PfNJdrvwqJQOfea4RUUMTXtdKjRDaohKNvBvDdq5QkvjsrCl/Wt0BDa05/lkq+cT8NMw7fwwtUC/bKcq72KEjSvyQiWJQw2Jpmph'
    'w4y+Qrp6gBoolXEXFk1tlrkAVYxPxAD3qEiyLC/wXKqwhngJMJin6ksuUPYelFU59eL4rOlCdj6ctV/G7ATXMqMO8lwmVIo8zMhm'
    'wOGch1c46KC04JoW/PvapH8GVqpVIlJeCKulANGXrVdww8nAVzgancjreczrcmYSqRIuDNdZZsJZsyGkwWLUbwDm7YIosowXxgi7'
    'zOKpQ3GeS66BZb0ZeoE3WObYaxj/4rnwscJrUpJ7FLFRwVhnc6VY4axVXJtw8U4f1KFk3Suo+zEs0w3RQp0tpsRn4vW+XxSKJE3Q'
    'TFRWKuB5X/5egl49oP5t1+ze/7Jo5uQerzW5jkpU0kKm1kDu09VRyNPeLL7BHM7zWYXaJl/Hqip9iVSFqxiVkpQoA1JYIRPDDYSD'
    'Xy95e403WMnRbjUHcoimy9y/gJKDyucof2nGdZd12OJ9HwFMpt3zvI/rAGrT1uI3yY8opS1SCc5R59KcpSz4avAhcgKPZj2va7py'
    'XxuN3sZnqCbe/PkX/gB6INoXdLRNTEKES9CSTItEgXSas2BCxDt4Az0GLlG+l/VDGnr7iZw0ddzpLXJIDCjOE5uxNPzSFcYaTmvg'
    '9xbk9JP/vEKn2Gcrbo2a22aGB2g6K1F6EnT6ZYwXkGqR08Ins9rcaM7D7PXFDoTH+sbHVnT/fDpT5GK8mMB0G1HbUo19Vib4MuBj'
    '8rVu7skPXwnig7C+EuQ5rSumVtdIQZWiiRYaKAsvwAo1kcQM5qvvqWo+ISPTzGbLQjiy0y5lwgUo5CmCbUtBWOmZziig5gmx/S7k'
    'YOp0T03KyqfdejHgUzb3mnoK/54DujQxDjxkuTXUoq7xibCZCHO2CW8wNboRhmw1rRnjB1oVc2kTNM8hQwFLKd5f5X6JxRfYg2nX'
    'N5BnXTYfqi5viI1UVGZJnmYyZzRxQvrqD9qbz7egAx7T2i7I/rxyD5nh++oO7ZSdT2Q0g3s0NH0ydVn5DPGpzzqG3mauLXBPwRmW'
    'aJ44XYS5jaVgsJYCe16wXaLw9/nDBwvwlbnOBy0DxSDrjQSpaApM8yzNcq7AbmD0Kgw3WHULSnFcrAsF5AvcTqHuBDpKtwr/6y9U'
    'uuTaMMaTAhLJLXUbJGsP2OGOLVRlDWS/XFon3vbyJ+WH76Zx1Piihi3nM8H9hUGxkpP97/tIxh2Q47p/IAR1Cmdo4ksusxSCjtqH'
    'iAnse8/DnKwc5jHuADoc5/OZd6X2feKq7/pw+hTa/ArjCr8TI5pzJkUh0YlDl4fnPCya+4MPp4peYv7w5R7dfa5uyE69mPhI1AP4'
    'i0feGD8WEh8JUyxnzuRU9GR/ScpVH1oGS73dGz8k97SN97vqZn49jkgY0BrvNvp20iVpUagwo2tQAjvaM4DJngOYe01lbyu16Ko7'
    'RjcLcjjHK/TN+QB2O406tiJJuZYsM4mlJrxvmwH/DsY+Lyb1QzcWwIMCN3GRP4rGgxSZ9cXYmRFFD87WIgbOXc9byNN1rPnvj0wJ'
    '9aycrjgMRHyNrx1VUqqUgsxsap3Rtgef78P/Hfv5HX1F35fge1wHBJAgta/ng9QYFe4WEEIaKKSAFrSqfbLgYpmehSqqfyBTorme'
    'S83SVCvON53FNSCDpbXtteovvLht44t7dn5CRQ5jJAWAyFIfSS+EUuHc0XUog72N7n/p4oNdy6azhux6WlHO+nTqF8mb2wQVjX8S'
    '85bEhbq9xf/UX8anzGYp/s9wymkqgqGBvtS8JWaoU4kUjGpvJz8VcnbV795df0NFTJoGyyFNMpZCYcCEo3lR8IO9De0rN0MDor55'
    '6Nt1MvcdrDzgOVoXrQ8meouyVJPpNjkrK1z0xl5DlF7UhVFWFXkqC5bxcDAhgpaXpKxfDdHzSoiVK4ESYzRWi+036cvoLVBKyWf8'
    'VAzrNNMmSxORCWNMkZow6zW5rebTDn0NOHkGH+7+dx7v0RxmM9W5wKPSZyccVujqx/DI0SJwpuCJhFxvqJ7ajDfUJV4F8aYvGsJd'
    'FrCPNsLSScCTBQqvFl6xnQp9hqgyHWoTW9ii0FwzpXvx+kTG7ytEXL2hYqBXw33As3QC9+W064eDH2991V1U8M9CnumC8sLSQoY3'
    'NIA0mL/t26qgQGhvl90sJg1Kx2b5GhnKHV9vM3DGuClU4ZRUGVNhGfwe4GBNGvdhWqJEg6lfu+/NQl3D4y/PNuTn/XgTFnKWFFpo'
    '7jTLWLioP5KAwRo47jf/8kFGH18bKV8/HxOyRROCoVmegy2MFuGD+QZjsDSe/Wamuk6P84mOyQXRUFhQIpdGu9SoMOUrnx9Oyc99'
    'Rtmtl7a7Xe3YY7QoJqXPQGpNKtBw5TmE86fCWIOFYFYgOmnu3eZZZNtcJ5RAoSBZkRhgAD3YWQrtZ7DhxNwzxgnclTM184/Yz8/Z'
    '+2pX6ZjnVuvSwhrNWcFT5oq8B3ObgAd71zhQvhdqV/UxQ19mT1Wla9q6jHFhjVYqLRRlQtPCZEF/Iow1mA94YH1ZDrTdjX3uPHcO'
    'jdcfjxHILgUjKvynC5ngDVNUOalUmMMY+MEu3wvUPTVFYnysucupQOM8aid5XiQO/d/CUfQSe/G5BnGwLNkXQMc+ZYQclj6E2vXr'
    'jVfFguaFyoClOTr7aVb0YjCI+8FOQgE2z6F1YKKsQ+ZQbDLBUJspWvTj6hHm7zmG5+Nm1ly36nYcc82SNM0l1YmhaCeB7sfHM9Jg'
    'ftgKwLrC2eOlkvNpEjEGB7CcZZlWSSIKnYge3AXBrx7QB1N4B9YXh52XynnN42OLQKBGd34a1UdTaZ96kTBNpcxlGmYxjDaYXfgC'
    '5Kypt8kXIOh4tnDdohOIvymhjnoAtcarAeQvkVm4hOs17lrYwZ6uD+ydatGvxa9ed6l11R2U19Ng3WuQOcdUIaQF4YzMkjBz6+Cu'
    'ppvKX3ubJXI9V+d4GcjRop513c3RYEA76DSuNTfa+ZIqQC8MaFow6MVeEHc466vLDD3yjdvKridVF188fW4PiXc9RsIk1jKaZNop'
    'rq0I5pX0gh3uHq6mfu4CQkeVuVigWlP0oA3LKRdhmbIOZqAQ1kF1V07JeQvTqkuM6/JFPkN1Sw7Vsuvscd0tYv82b8qnpxljHRUF'
    's2GV/Rb5KgA9WALBS8zHs3EJtY3pDUBNrjizSeIE44mGfhy+QvtgU9EVgVIvVNebp1osa25ePAv+ojWZmsSCBJFIAVKFG8FFgA+W'
    'BnCAfnFJLufk7OBnOZ0pv8bKO19fGkAvcjpFm2LZrifGfAZRWGAqzZDfDTmIz+BX76IHXgp6+griOX53MPdVHxP85rYP7aKEO4G6'
    'bn5u+0QyWEkiW3bW2Pe1/1Oy5fM+T3vfW8gTAbnzaVB5osM1G8/EvKClDymBBemZVy2eY0wHd6qGqYFl7dzpghxPJk07U9XLcOIv'
    'Hn5jmZWZLhJlM5mkYWXzC1R80LF6NmsP0eLyCYO3Pi0AfYRlcxiUNo39RHbxNjb1wnf/tf3j1hYN+IxxURSgExf0UfojD2bDH1Z4'
    'mO79+JdTZXx07OFYde0tNlR4hx7KTJ7J1Nfwc6lk2JgPwK6gDpdv4adhnXYFkz7WgMZYTbY4pQW5gImazqAF27/VKtUmd1KB1JkQ'
    '6EGHGewBOljv/MPWN6MalbWadV0RviyLSNCiqaLmRIk8UxIkA5XaDZ1sNsINVvPwGsU/px+jkvDlBYHfR72OaUFTxqjG88qM7ctr'
    'HyoGq3I5bMEfl6aekof8v+Z27OvC0Ipbvtb9nDXtMjGmnS0bym2ddknCQHb6210Zs8ZmLgUlrWKahxdjAz2/v0PNYJGWo26k018L'
    '8pO8KsjfRl0xhcp1QZDjGKtMFFpKXTDpewGFo0hHv68Hfok7WHjsSE26sSiX47ZBhYiqrkHznewFSkaD91r62hbJNS0KbcK5yBvh'
    'Ave65/YlK9sHbemr8qvy3ueCLNtgXyDlna8M/VOshfNtomXqQNIsDb9lrQG8eok4mHO+HGFIjtSibpaBty6RezRv7+IaNVBmHPIl'
    'FDBjsnBdwDuAgwnjI4SpKvWXz/QA8Knpje9YYmJYypVJ80JR9F9TVgQ9nyDUYC8+n3f+GB1cdJf16xTq8l/K9yJt2uauk/ClBrQ1'
    '6+uoN23gJhVJzkGYIkmDnmtv6MF6FX1Ge1Hdd+mTXVHe8fRhHMyHUmxpkVkrFU3Qu8uEDArMGPRA4CWPruBd9ijzkGhPNnOfhRfz'
    'oqCS3HFXJJQya+kG1t6iDKbjPquu3Hm2WL5+7lhfBPCyXX+3al4V/pz5wIcl36H1o6WiohEsczynmmoFuQw7573JeUvMYK7KKxpO'
    'fJLSlPxxG+N/giucEWj5FUxk4TfZINRg5aufoW594ZpuOkPuFAFLn7SzT3Ym+JGYvD1jNSitLKMcj2swJ+U9xOH2qZzOOqVkgcxr'
    'oiadYvJ2tEHko2YRNexPc6/blUiNstwGHcoeoB8MGq3sXfNX2T2rXT7Ytujl7Y3xX8e8W3LKCp5YybhjWuVhxsJgg/X+ObZ+yA8a'
    'Qn9BtU12/rj4duHHcx7Xs6ZLmvujvqn9MBw/IOUvqAmPKsbgHKQpMsoY80UZIUZXiehHwwcjBM9m9vFEXXdFeK269r7N7+TM585E'
    '1c6LLLWJTRQY5QDC7TXDWIPpjrcQXvV2ZcDHyxrSaf83B2qkMUxnyoJQmvZn6y3mQP0JjxfQ3Kiuj/zktlo8vmnEaD2nUpEJSBzN'
    '0DUKWqIPSK9w1rPR843r+YnrC9lV1V1Zb5PH/K1t39yR7M7rerGNS1bW3rj444zsH++gnj07IPs7f8bcvEJIZVByCoZqQZigf/RI'
    'yRMhz3SEyRhMXXw5JyP1sxtJ5JYpuk1b2a5g+6Du3hqXdRN+osppacYKqv7HV1CmJbq82rEkyYqgLfdLRAz0UPjFJ86TnXaGUv0h'
    '1bVrnLHMcJ+h8orZc2nQ2+dSUqocDdfphEGvnlCH2+IObLfzAh7iZFFMGc654obbItUc2drM1GucAdlAmsg+tPOHBhLoe0/J1l5j'
    '1PKPiuzUeHgmxKH1sSzYZgU5PD7cieuXJFMUTSAtLZzWqdrA7zqCPD2/v0vNYDGqV0R0/V7QRul+jfK7lENDL9cpOoxFxpTtyfVr'
    'wL9nu7usKnU/HZOsSLsLc+eN56UvO5rNbdm8+ygWaBNV5DRzSVGoHO2jcH/Bj5EzWEj6C9T+Z1rEuoW/uvRwFZHDIBMts5w7nSfa'
    'UBvsCRCAGSwE9AWmuDoKF3GZITFVukJfIcY1yQvrhECtIsEBTdIwKwGogP7ouScpf8kMkC/LMqzpuOrmcy37RpyXMzN+TNyMs9xN'
    'lhqaCimTAj2wcLlNFPxw1/M5K8AX4M+aMmoEr/QTI32OthMs/E65BmQwr+PFt0eqmsek72aMFyKXmnK8RyKcKbMGZDC3Eb99hcrm'
    'BtorPxwB7Ycrn2rU1cP5P7VXn9Xs6lt9tfX5v377NLkVMRlcQoGSMmOJtVAwuoG/Rxp+70NBICDOogPiCFyhqvVxop/k9Ozg6+OI'
    'Ei+ag1U5gdGJCQWWmpwlTCeQbTqN72EOeL3G9XOd7rJBwvHk1o9GrXzwCG9112Yf9W4b0wTeKQ4810UimaRWqTCvUfjDKYamRqN/'
    't218o4Gu//Xcn64YEzyRBRVaCKG1T/kIc7geaThVPfdZNMtsad8WbKrIGVy3Kiq4KPAm2kwWVpjUhsVMCCoQdOvpLYsVd/mF77U8'
    '+HdRudhUJDaDLNe+rQsz4X0JIA13teb1DerLf55sk9epmjvof898ZcyJr+jca+o7PBpq5ptDLSPtUW/aEvktlI++ySQPi9BHcn6B'
    'msD29tzdlRV5Wa/wuamWHQMeukRcPneJ6J9LKlgqTJ6BMs5CHvaE+2MP5fC/rs6oZ77vTJdVgbj/NVd3UWPCjOWMslTSRDsmbF9G'
    '18KuZ7HnE1weZPGxB0bbZUlsnf9xcf5tdPCQG3UHEyQkyjKlzEnnmEt0DjlN+7Lci4yAfdTzzZwnwUVYtjD+OPMu5XiucyMYFDQ8'
    'RiUKfkDhtorqDRMNPpDY+TxNtwd1bM9sK31pF08NQ4mm+/EbRh6s0PAB8LKcIGC17Pmyt3N2SUaX387J5ecDcnhwcHJ8dvTgC3VJ'
    'jj+a9iYifmlB0SylRlKaAXtPiL0m5aoPLcNFgtYQsSzcn8Cytf5SrLaoZNB0u61gFtWn12VFbnQBrivtsDZmMd6nYyDJ93Xn5OBb'
    'F5ZakH3/YPUDAsHO9WorVzrPs8wIhU63DadIB3A+eI+fD/dXNVt4v719aOpcTqdg0SI4KtsqqrxICaq5y7hUkKfB6NZGuMEK3x76'
    'SPpHt+WT/sO8QjwID4XWl341q0qR/yyv6/5pwQKFsciKBO1Lqmi412MEAR3+YGL5aYSA7yoZkyabJAWgN5PlPteOB5XNy+8P5sJ8'
    'LcvlnLejpuqfb5/QIs9zqqwB4fO1g0S/+Ppwa73wpXKnZd1cz+Gx8st3wvIvLN/mM98jctlOI6YckeU5GvNKW6v9tNHwTXqLfhWE'
    'D9g6PLoRzonSbVl3DUy/gKqa+TQiJGDAFoUQVGYq4WFHZQ3GYLrrRNkFOVLX/jTsVPAvNB/aGCM8z1PBgRaWFdxYasI8rMMZLDVl'
    '9fP+EfeimXgnJqKXK6Op5AJtLa9hw5ngIaBAfC2LLihcBThv/IiFQ2WiakBzDk6kWkPGkixsLQSABrMSV78/ms1vUUoGqj6DN9/3'
    'HmROyQRtwDScaxlCGshtffr8tn+MsT5EsNc0t0vfAsV/1dw/GJc7XXsdn+u527RR+YhO4PVhmeB4k5zM5fusRpIyWEL0iR+Uvg8V'
    'uehCJSP8IWhnvlBzpGwdmLO4/s4lHK+a8j15U6GLcI7b+5ADisMXUKg50Gr2LtVTg5qYQBQ1mlounS8O0y7vyd860IFSiE7U/K77'
    '8mXbLHD5yruuLUg5IdOGzMoW7KdPn17ktk7naN/8BW1EpDuBIit0LqT0b51MhdleT8zVS2rWEDPczUYK3ia4PI7lm/pOrj/Wz/Ba'
    'z7qxKnEmYWgQ00JBtpH1MPDVE/Jgttmyz+GeulWVLbsKKveYyhyVkiK9hGJplhYKzZdwD7vNeMPd11cw3tX1bfS7meyoA2xELVie'
    '8DQVDJTMwYVTpt6HHKyq5qQc+VzQvbbxD5nX5LCEKsZ0zriyKGQdaJ65PAkbmGtxBgoMnJQVOVNT8k9f1V/hbfdO+0WDttOznDlt'
    '7kr4rbsRfgKcH4mBmm3Rv0OmZjyxNlGGp1AURdh8+wgxw13G0vc5g9tO+v3zn/+8PDi73Nk7/nbmMwkeOlX42EbM41tW5ILnfuyE'
    'zVUmNy1AL+jhrmhZ35Q1OVftTRd87wKwB4i+ddmgIgdf5u3K2ayj6BSqCmpyVPp64fLndky4tMgsZSozNAdjw4nlH6FnQDOqvL2d'
    'bvvyLj9O7XBe3yxmgR0PNCYHyNCyyKhOtEK3P8wuAr3FGawBSve04M1MP8Nw61dKy53NnM790AJIijSckxJGCpgDPS9mSl8yg+cB'
    'TctDb0V3aT025EmGpK4sDDO6ECpTGQ9nWW8AG6yf0GsMf9aPGutCMwtCh824nBY6K6DwORtpX5ZewgUUSk+u8hWu5va+vCbeXtwF'
    'mI0Rte5Od0sOqjKuC1tRiCxjaKlayJwNRpfehRwsPepkfoMK6qhVYzVBhJz8CZFzXIwFI2yWMssS4BtYWgs0mOA//XrpA/V7lU/D'
    'Mv2DmBkTSkLurORW0SD5rz4/mHI+Vbb0Gae7sHSroXLldBzVGNQKUdhESusHagatrgDQB+9+vsJIa5rpgyNZzWczb63u/HsOFRDW'
    'zUP+RH4sUx5GyimrGrK1V5W3gC5XTE6hRZ3LUDoIxiBTiQrzu4Geq3cJGu5ces1/0dTTbvLYH7deH3aacWnytfO68YlWMY+VLkV/'
    'CB1eBdJKu2EB+gEPeJZb34ElfWw1/UPVteoGlUZM6iv8kMmUZZpJmVG5gbsVrKsXYIPpsxUQ/0I2fe7C1z1tqhZdst2YPC9JFWeZ'
    'cdJliRRJD+Y24A5WJbsCdwqTpi0jKtdYlrNUaRRCwJzlugdLTxhD3rHHj4/m16q/I5dRl4MAqoXI8pTmPahfAgx5Z6bjycMQUT86'
    's6tz9xWoh91D5d7n7xd7nw9GKx5j153NVxzEBFKYMyZjOmNpahnbxGg/ctZQ83esiRffr2aSHF4cH5ztj142fCbfDg+P9453Tvxf'
    'R/UnSil3QkiFJ1gL2WNdHkn6PYKgwZ45VujopDg6JWXVzd/4rG5vy6gMWkBHjgLXSYb/Pw1nvbyPOdxFhusx2ryXnuKutPzsW5Rp'
    '4GfuQZ6hQMpMGnyQXocyUMgbPXjlewrims3KuiupGY2b+2XgGaI6vvtU9YyqTFudoM8a3p73IIe7l14LPWaPeGfrm/FDbfyhGI0X'
    'DwREPUO5DDLLHHdpYkxYKPXCHZjNZlo/NPW+bJobskPOfcef45oc6/IvRbZGAHoZpvmNbB38RF5NOYt4t0gdVQ7SVMuiSKFINjP/'
    'i9QMFhs+bf5atsVEg9I2ZMenzlziP0Zo/OUGPYlltUDEAhjmsrzAvU8kZ4kOevn9oQdLYHqCHC0mt2PfDvKs+UQEJWVNjsjES41t'
    '8vUTSVNKtkwz8bltEDV3CF1lVYDRUlnKU/Eu72FCHul4JOOjq/Bmzw98w4GbCvwvZ8qMZxNUczdL/nlKjr9/Wu7MJ781cL02nyPg'
    'eyeKK5pmeZYYycLDKTcSsoGOwUYCnalmUpKRaWbdXtwCmHE3DOrh9btS1pZRL/Aml2nBCtDUKi3CHYH6Ig/mC5yBf4P4vZttgGKm'
    'S2CN8d64SF2mnUOXICmCd3otymBWxBnapRX53CynUHRpf93LyxoTen0NceC4cmcSl1NtM2o5DzMXBz+Y3jorjWnQSDtX1/jDXfrj'
    'nrpFJOikBo8pDNRcoGY2wHLhdJqbMK/vYQ7W0HAN1IkiRk1uVY3HKWqyr2J48bIUZY4rmGMR3L2E/GAT9GKFudr3xUeAr0rjx9vS'
    'd4j07QSO26YbDT1r/bU/2I9vDZ4kkBvNlaaMUSqLMLe9aeimb27sEh7dqOWsIfvNXM+e5pTddo1NB+gFr0Fz1BBZRjNjkjxYXfeS'
    'gqv3SRgs/xaRu6ag+OsJ/iz5dgtd08xOdnR/oxfka1N3umD+1/wmJmAq0sw3L0vAMWlSVmxgP5aI4UTXfKLJFq7vH+dHFzv7B791'
    '7eCeXk5jsqASwVWR5oCCWgkeVqwe8RnwBdpg8cNvvlVT2YLpivl+79o2+VhK5Gu7yxWDVGvNUkVNeHjIRrjB6tHP8UcM2Zl1nth+'
    'Oe2Kfj6X12NUebdRs3BZoXJaUMlkLqhhQSXzHuJgx3AN0PfS+I536zOtA3aCcWlhEllYk+tUZRFsraANyNTcl3LP27Kb8Xla1n7Q'
    'bn8rHcD5h3Lq0lwYEfbTQjgDFa+el75B+D2o2X/6h5yZT67/0bTTWX9GJFDOVAo2RT0YTu1/RnoFNJiBiu48flb5QcZdWURj9QKW'
    '9aHLdMzL8fyabF2oGVhy0d9ClTJPVVKIVOC1giJ8oSLxB3OsV4C3cYmVr3nu6sbmtauae2jJ1ugWlWvbxXvr576E3X+Nius64Lmw'
    'TlqmC+H6rMQmen5fQ80HK1hXjnZz7+aIavHIjYnGNd8yjXNIie8ptcDvkDEou0wSW3YhbRoNUV2fVCKyLAGRp/4BKgmvxy+SMlCO'
    '7HmL9vb1PIY1YZnIE50btHOpDnd2evr0YJXH54tm1pLjalGaMbk0Y1XeNHfTmy5r/N5P+14WRSZy6Qsxsq/qKZBa3TZRPS95mmaK'
    'c9SVuRXhIdUfIiewKD1tHp69vyhnvvHoV3U7m/ou3Ptd1f+Df9G9qZHzKkInKeEsp1KjKDe0iF6SPsQMVrV18YnslbOuXZ7voWPJ'
    'jm/V7W/QjlWTbppwHZVFlFnLZcGEcCoX4eLVfrgD3dwL+Pe8fGiDp5YRnS4RG+Wrf2ipUJYunaeDuKkKQLMEXXRqU5sl1gS9tzj8'
    'wVy3i3LcJR787m2Em5XYe7fOaIHfxJSFGcdyS4UyylgZbtvWB3SgfOqLxtoFuSiNGT88vuw2P2O2z+ksyQ0YqpnNVTA/YD3OQEyM'
    '8CyMJmVX07hfqklT26jZasowmTifpg9pIYNm1VqYwU7a6tc/N/fo1XUZIaMKIKY5u5KcJyZXNLGS03Cr4Y1w65nq2QZKrmXpR1v6'
    'gerTx2nVP3wQdRnlHt2iS9tGPbAUqDHRVeZJkiZJZnQfLh8ouNpIwmBK4Ql5G9VxO1mWq3tF5GMuXdKFL6prVX0dJy6FTJUrBGM6'
    'YTL8rPjM+Pvog7miIzCtb3qrWtvls/rJtMvUXUR78XdbJ+UdEPTAvpaV/zOnMZtPU5dza51vzaTDEa8PkfPBuDZbWZQKUJYfNZPu'
    '7f6rf9g9mCw34WtZR1Y5Ks44Y2lhCt8ALmws9kL9kPgtQhyeNNNlMttl8zhUNCYYpn2ejiwsevFqw2TPHqADRa5HY3Vf+5b6tuvN'
    'dlz7iMFuFTcGmYFNDHoxBRSasSSY7bEBbLhr+hJjNPPdRSNyAnNbuFyCS5zIC5Xanrw84QwWZ3kFcNn6mI4/BLt+3HfU+A9rJM2s'
    'KwolUbMUPVl6g/i3sLbdNQOsfE9A/ZC6N4K6QXWmYvJRcL9Sbnwef6ZNvwMYxv2bGH3dbeu0qafrdzJwNHNjeEq1pdYIE05XfcFl'
    'CHS4C1d2rc3RqLf+Ba6NqpDNjU5zzkVOExfuUPsGYrgNevgyqFs/0g4VyTTw1L3M2PKpWz5W3H8YqOMJ6Cy3Ik8SGW7M8MuEDDYS'
    'ZElBV4bkLz5MY3oNOxCFSHA7Af8nwk0Z1oAMeBDRaekyB9HwcfP6BqoHrFEzR2/Gx0hG6EuvL1Fbv30qyROjDdOUapnnLMxXf+zB'
    '8mlH3uJftrO47SJAy5Y9bVSTaArCWZOnFMBlNnw+g1iDpek9muzTx0rTs2WT8B6DeENZes4oFHuQprnkCQ/7vn2RB3tcXQ5f7PJ3'
    'Dxa+aKeDvSyvfdR+dX51dMJAkUOeCZpQSArnNtjQkRQMJ3Ifw7nbr+K8LwKZD11l/Q9HFfRJ0FowY3WBPmQ4uvgcU+5Lw2BJapff'
    'zg5GZOdsnxx7wJ2zvQNy+u3s68GfMdniRufW95HjTBdJFsw+3gA2mOK4RDlOTs2F8gH6BZqLum1wYSdAXNlGaRGWuyTPlE0oFZkQ'
    'QWH0HuJgZ/VSLSq8IqP70vn31N2qG/R8G9dlChKWgEmtpoliaXi07gawgcLcrxBOImcEq0SKVOpCWZfTNDwacS3MgMftxdd/+KDG'
    'dPYwHj7GZHHKppZqxiWDItM9uXmFN5j18grG+Dq5a1VHJQY6pYAqQDfBMGp7MvSENNyteSFSO306n5lW+db12+T4+NMy5z3GSslx'
    'k3jqEpZTtKODaRybkFeAP5b5KFY2DXzgH1dy8jjOzg+S9JbSt9rPNW/9yLQjhfq1G5AdlfinlBRSMmOEEGkaZvkNCVfv0DDcTfQP'
    'HWjFLyeg7jUTP3nyGgL9EgJZH0aA5VSg1yBovoHNDVhDSUcPUeFZ8QaRJeeglivqCwmPp91pCnXlC7SXh5w632DFcJu4sDHSF3iw'
    '8L3/InrTZT2d+G6G3Yr65wIf6zn1FQpdqdzT+PoYj77wZSF5ImmqjQk3Ho+lYKC3tHWwVTP1Jm/nYi+d6+XU5qiJej5uzYsMfSnK'
    'w93rouCH0yxvUfcfBwc+JFgun17VIiLkpizNpDVKO81SJkwEz/tPEwTfwA+ng1ZCDF1JIf4bP1irS4vDI1ZVPs00aDQEKq6ZYgVV'
    'JrMpbrndeKF7g/9/YbnLg0ZZEvda4URhLQNwhicbGtb2BB5U73xGh71L8G4ByHKo4xncz3BZPzf3aNEsWwY+TYhT921U3McivyLP'
    'rclz1FAcNrH+i6R86Bn61fX+Du2d94V2Sx+o9/mfeLkey+6ihp9TyLh0XHBWaLFRG2/GHKx8rXvLBripbecTlctRq8tx6zFnmWW0'
    '0EpQqoEJlWziLAg3WPjnBYrvPqubxYMQRJv8/LHPydOfYpy0FAqeSK2dlCoLJwpHkzCkPnpsejUyY1zhyquFkzIqVJAlmdJFgfdT'
    'SU03+KJrwK4e0AaLXF42N4uGHI2befVUefJH7adRVr7u5PIr6fIwTjop0UzH5cz3cL2et1GJc6DSDDJ0wG333hrk+NeJGciQ/u4H'
    'fU39y1pbzZrlWMRLpGfatQksq6g6C5HlyG7uDEXX1vHgy0IP0MFO8HdfP9cVtI/AW6iWbPnEvIVu0BXzpvqU7NS+j36l7uM9v1RZ'
    'yVJpElskmptgBPdXqBhuCcq7rqnroc+qHoEfPDslP8p61hmzx34MWN38tk2atkTlqCri54Kh1fOJHPb3DbM0Yw7vOeQqx1+DC7GR'
    'lqtHYtbSMpBj8QNKVPRHeNvmyxF3s3FJtrqXOj8+4lD9q2zVb8+NZ1ZK8WLugg9uK+Ap51IbHVyQj1Az2An5Uf5FupEfrms2D8sM'
    'lp1rZO1xdlrrKzbP57Oxbw7oC4dIvnwv6++GiBQgLVyhwI+yT4N6/UPUDLcmTW2hdSiZuyrF35duf0wqBwoFmRqVqyIzyYYTsB5o'
    'sBGYr9rhjnb2I7ZM2KSwnIksM7kMNzFdg/HBzpHPVSt/gn/Ss2qx2h0VfQwqoxohKG05tTaznCNPQUm9CW2wEvM/577muSbH/gEL'
    'f9k7eFL/n9HaaVrfbOYmIjTHRaoKo9ALNBbycP/y3sCDebr/vfOnPxHnxycn335c7px87X/4CqYk1RaERmtKsODhewsxUNrwf3sJ'
    'fKLaKRK1/Tih9wzu/FCRpr1ezuCJqkPVRlnFnEsZTWQw9rSKG4QdTND9N1jbmSY//GCjh9RF39/wIXvlOb0x5nFXGUWlyi0agriN'
    'wTy2CPDBEtw95rZ/XgBc3aZty+m2j4k8PkOcltZWEPWQTQ36cHnCTOFc+IG343UVdg3qYIEXdXJ88Ff/uwaZFCJj1DgjtQsbssvP'
    'Dkbl6kjH7Ydoapea8zSA7ABF8DSuJtNY7jMo0fHKGQUTVLwv5km+xX5CHkwWQju/fSiZ93WgvkhqVpobiHGm0arknLJEKGVzCHeS'
    'DGMFtFhPdpLnyMh1raa+nqEkY//IP9/G31U+5XnZPrSpyrtSkUa3pc9sn3SG611kKxyTOcpzYwoJGeqCELevSVlPydU6Ugbb3dk9'
    '1LNFt9q3ZdXMOnMevCvfOD9t2aBnEzNFjSs0vDSjqeZMBqOavVAH0xVv0T77Al6oHxtFjeboplggo3/Ple0E206l55P+O25cxrhJ'
    'HKQ5pOhU9ue7LyVDGKb/8b//Bx+kJnc='
)
VERIFIED_LOCKED_HOLDOUT_ZLIB_B64 = (
    '...'
)


## Kaggle environment check


In [ ]:
def prepare_kaggle_environment(input_root, output_root):
    from pathlib import Path
    import os
    import sys
    import importlib.util
    import subprocess

    if not Path(input_root).is_dir():
        raise RuntimeError('Upload this notebook to Kaggle and attach dataset/model Inputs first.')
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    if importlib.util.find_spec('torch') is None:
        raise RuntimeError('PyTorch is missing. Select the standard Kaggle GPU image; do not install CPU-only torch.')
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Kaggle Settings > Accelerator > GPU T4, then restart the session.')
    if not hasattr(torch, 'amp') or not hasattr(torch.amp, 'GradScaler'):
        raise RuntimeError('This notebook requires a Kaggle PyTorch image with torch.amp.GradScaler.')

    dependencies = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
                    'tqdm': 'tqdm', 'IPython': 'ipython', 'scipy': 'scipy',
                    'soundfile': 'soundfile', 'librosa': 'librosa'}
    missing = [package for module, package in dependencies.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        print('Install missing dependencies:', missing)
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
        except subprocess.CalledProcessError as error:
            raise RuntimeError('Dependency installation failed. Enable Internet in Kaggle Settings and inspect pip output.') from error

    for env, folder in [('NUMBA_CACHE_DIR', 'numba'), ('MPLCONFIGDIR', 'matplotlib')]:
        directory = Path(output_root) / folder
        directory.mkdir(parents=True, exist_ok=True)
        os.environ[env] = str(directory)
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print('Input root:', input_root)
    print('Training output:', str(Path(output_root) / 'train'))
    print('typeguard compatibility is checked in the next cell; Internet may be needed for typeguard==2.13.3.')

from pathlib import Path as _QBHPath
from datetime import datetime as _QBHDateTime, timezone as _QBHTimezone
import json as _QBHJson
import sys as _QBHSys
import traceback as _QBHTraceback
from IPython import get_ipython as _QBH_GET_IPYTHON

_QBH_LOG_DIR = _QBHPath(OUTPUT_ROOT) / 'train'
_QBH_LOG_DIR.mkdir(parents=True, exist_ok=True)
QBH_DIAGNOSTIC_LOG = _QBH_LOG_DIR / 'run_diagnostics.jsonl'
QBH_STATUS_PATH = _QBH_LOG_DIR / 'run_status.json'
QBH_CURRENT_STAGE = 'Notebook startup'

def _qbh_log_event(event, **fields):
    record = dict(time_utc=_QBHDateTime.now(_QBHTimezone.utc).isoformat(), event=event,
                  stage=QBH_CURRENT_STAGE, **fields)
    with QBH_DIAGNOSTIC_LOG.open('a', encoding='utf-8') as handle:
        handle.write(_QBHJson.dumps(record, ensure_ascii=False, default=str) + '\n')

def qbh_stage(name):
    global QBH_CURRENT_STAGE
    QBH_CURRENT_STAGE = str(name)
    print(f'[QBH] START: {QBH_CURRENT_STAGE}', flush=True)
    _qbh_log_event('stage_start')
    temp = QBH_STATUS_PATH.with_suffix('.tmp')
    temp.write_text(_QBHJson.dumps(dict(status='running', stage=QBH_CURRENT_STAGE,
        time_utc=_QBHDateTime.now(_QBHTimezone.utc).isoformat()), ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(QBH_STATUS_PATH)

def qbh_success(message, final=False):
    print(f'[QBH] SUCCESS: {message}', flush=True)
    _qbh_log_event('stage_success', message=str(message), final=bool(final))
    temp = QBH_STATUS_PATH.with_suffix('.tmp')
    temp.write_text(_QBHJson.dumps(dict(status='completed' if final else 'running', stage=QBH_CURRENT_STAGE,
        last_success=str(message), time_utc=_QBHDateTime.now(_QBHTimezone.utc).isoformat()), ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(QBH_STATUS_PATH)

def _qbh_log_cell_error(result):
    error = getattr(result, 'error_before_exec', None) or getattr(result, 'error_in_exec', None)
    if error is None:
        return
    formatted = ''.join(_QBHTraceback.format_exception(type(error), error, error.__traceback__))
    try:
        execution_count = getattr(_QBH_GET_IPYTHON(), 'execution_count', None)
        _qbh_log_event('cell_error', cell_execution_count=execution_count,
                       exception_type=type(error).__name__, message=str(error), traceback=formatted)
        temp = QBH_STATUS_PATH.with_suffix('.tmp')
        temp.write_text(_QBHJson.dumps(dict(status='failed', stage=QBH_CURRENT_STAGE,
            cell_execution_count=execution_count, exception_type=type(error).__name__,
            message=str(error), diagnostic_log=str(QBH_DIAGNOSTIC_LOG)), ensure_ascii=False, indent=2), encoding='utf-8')
        temp.replace(QBH_STATUS_PATH)
        print(f'\n[QBH ERROR] Cell {execution_count} failed during: {QBH_CURRENT_STAGE}', file=_QBHSys.stderr, flush=True)
        print(f'[QBH ERROR] {type(error).__name__}: {error}', file=_QBHSys.stderr, flush=True)
        print(f'[QBH ERROR] Full traceback and run history: {QBH_DIAGNOSTIC_LOG}', file=_QBHSys.stderr, flush=True)
    except Exception as logger_error:
        print(f'[QBH ERROR] Could not save diagnostic log: {logger_error}', file=_QBHSys.stderr, flush=True)

_QBH_IP = _QBH_GET_IPYTHON()
if _QBH_IP is not None and not getattr(_QBH_IP, '_qbh_error_logger_registered', False):
    _QBH_IP.events.register('post_run_cell', _qbh_log_cell_error)
    _QBH_IP._qbh_error_logger_registered = True

qbh_stage('Kaggle GPU and dependency preflight')
prepare_kaggle_environment(INPUT_ROOT, OUTPUT_ROOT)
qbh_success('Kaggle GPU and required runtime dependencies are ready')


In [ ]:
from pathlib import Path, PurePosixPath
import os
import sys
import subprocess
import json
import base64
import hashlib
import random
import math
import time
import logging
import gc
import shutil
import zipfile
import stat
import zlib
from collections import Counter, OrderedDict
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.sparse
import soundfile as sf
import librosa

if not callable(globals().get('qbh_stage')):
    QBH_DIAGNOSTIC_LOG = Path(OUTPUT_ROOT) / 'train' / 'run_diagnostics.jsonl'
    QBH_DIAGNOSTIC_LOG.parent.mkdir(parents=True, exist_ok=True)
    QBH_CURRENT_STAGE = 'Runtime startup'
    def _qbh_log_event(event, **fields):
        record = dict(time_utc=datetime.now(timezone.utc).isoformat(), event=event,
                      stage=QBH_CURRENT_STAGE, **fields)
        with QBH_DIAGNOSTIC_LOG.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
    def qbh_stage(name):
        global QBH_CURRENT_STAGE
        QBH_CURRENT_STAGE = str(name)
        print(f'[QBH] START: {QBH_CURRENT_STAGE}', flush=True)
        _qbh_log_event('stage_start')
    def qbh_success(message):
        print(f'[QBH] SUCCESS: {message}', flush=True)
        _qbh_log_event('stage_success', message=str(message))

qbh_stage('Load runtime libraries and initialize GPU')
try:
    import typeguard
    if not hasattr(typeguard, 'check_argument_types'):
        raise ImportError('CoverHunter needs typeguard<3')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'typeguard==2.13.3'])
    for name in list(sys.modules):
        if name == 'typeguard' or name.startswith('typeguard.'):
            del sys.modules[name]
    import typeguard

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('Select GPU T4 in Kaggle Settings before Run All.')
AMP_ENABLED = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
qbh_success(f'Runtime ready on {torch.cuda.get_device_name(0)} with PyTorch {torch.__version__}')



In [ ]:
INPUT_ROOT = Path(INPUT_ROOT)
OUTPUT_ROOT = Path(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
qbh_stage('Locate dataset and pretrained checkpoint inputs')
for env, folder in [('NUMBA_CACHE_DIR', 'numba'), ('MPLCONFIGDIR', 'matplotlib')]:
    (OUTPUT_ROOT / folder).mkdir(exist_ok=True)
    os.environ[env] = str(OUTPUT_ROOT / folder)

def unique_path(override, matches, description):
    if override is not None:
        result = Path(override).resolve()
        if not result.exists():
            raise FileNotFoundError(result)
        return result
    candidates = sorted({Path(p).resolve() for p in matches})
    if len(candidates) != 1:
        raise ValueError(f'Set {description} explicitly; found {len(candidates)}: {candidates}')
    return candidates[0]

def extract_dataset_archive(archive, base):
    h = hashlib.sha256()
    with Path(archive).open('rb') as handle:
        for part in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(part)
    signature = h.hexdigest()
    destination = (Path(base) / signature).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / '.extracted.json'
    if marker.exists():
        if json.loads(marker.read_text(encoding='utf-8')) != {'sha256': signature}:
            raise ValueError('Dataset extraction marker mismatch')
        return destination
    with zipfile.ZipFile(archive) as zipped:
        planned, seen = [], set()
        for item in zipped.infolist():
            normalized = item.filename.replace('\\', '/')
            if normalized.startswith('/') or any(part in ('..', '') or ':' in part for part in normalized.rstrip('/').split('/')):
                raise ValueError('Unsafe archive member: ' + item.filename)
            target = (destination / normalized).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('Archive traversal/symlink: ' + item.filename)
            if target in seen or target == marker:
                raise ValueError('Duplicate/reserved archive member: ' + item.filename)
            seen.add(target)
            planned.append((item, target))
        required = sum(item.file_size for item, _ in planned)
        if required > shutil.disk_usage(destination).free - 2 * 1024 ** 3:
            raise OSError('Not enough disk space to unpack dataset and retain training outputs.')
        for item, target in tqdm(planned, desc='Unpack dataset'):
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zipped.open(item) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
    temp = marker.with_suffix('.tmp')
    temp.write_text(json.dumps({'sha256': signature}), encoding='utf-8')
    os.replace(temp, marker)
    return destination

split_candidates = list(INPUT_ROOT.rglob('dataset_split'))
music_candidates = list(INPUT_ROOT.rglob('2_music_database'))
if DATASET_ARCHIVE_OVERRIDE is not None or not (split_candidates or DATASET_OVERRIDE) or not (music_candidates or MUSIC_OVERRIDE):
    archive_candidates = []
    for path in INPUT_ROOT.rglob('*'):
        if path.suffix.lower() not in ('.zip', '.bin', '.dat') or not path.is_file() or not zipfile.is_zipfile(path):
            continue
        with zipfile.ZipFile(path) as archive:
            members = [name.replace('\\', '/') for name in archive.namelist()]
            if any('dataset_split/' in name for name in members) and any('2_music_database/' in name for name in members):
                archive_candidates.append(path)
    archive_path = unique_path(DATASET_ARCHIVE_OVERRIDE, archive_candidates, 'DATASET_ARCHIVE_OVERRIDE')
    unpacked = extract_dataset_archive(archive_path, OUTPUT_ROOT / 'dataset_unpacked')
    split_candidates.extend(unpacked.rglob('dataset_split'))
    music_candidates.extend(unpacked.rglob('2_music_database'))
DATASET_SPLIT_DIR = unique_path(DATASET_OVERRIDE, split_candidates, 'DATASET_OVERRIDE')
MUSIC_DB_DIR = unique_path(MUSIC_OVERRIDE, music_candidates, 'MUSIC_OVERRIDE')
qbh_success(f'Dataset split: {DATASET_SPLIT_DIR}; full-song gallery: {MUSIC_DB_DIR}')
EMBEDDING_DIM = 128
EVAL_BATCH_SIZE = 8
GALLERY_SONG_BLOCK = 8
REFERENCE_FRAME_BLOCK = 512
RUN_CONFIG = dict(pipeline='qbh_final_v5', seed=42, epochs=30, patience=5,
                  songs_per_batch=8, samples_per_song=2, accumulation=4,
                  freeze_blocks=3, embedding_dim=128, vocal_ratio=0.80,
                  global_triplet=1.0, lac=1.0, lat=1.0, grl=0.2,
                  prototype_diversity=0.05, margin=0.2, mining_top_k=5,
                  memory_capacity=4096, lr_backbone=1e-4, lr_heads=3e-4,
                  positive_selection='same-fragment reference bag; use the same song full recording only when the fragment reference is absent',
                  weight_decay=0.01, clip=1.0, amp='fp16',
                  validation_gallery='train+val full recordings',
                  selection='fused_micro_mrr', score='0.5 local MaxMean + 0.5 global max',
                  test_gallery='all full recordings', dataset_policy='audited_partial_source_archive',
                  tie_rtol=1e-6, tie_atol=1e-7)
PREPROCESS_CONFIG = dict(sr=16000, chunk_seconds=10, hop_seconds=3,
                         final_anchored_window=True, bins=96, frames=83,
                         cqt_hop=0.04, cqt_mean=3, floor_db=-80.0)
RUNTIME_VERSIONS = dict(torch=str(torch.__version__), numpy=np.__version__,
                        librosa=librosa.__version__, scipy=scipy.__version__,
                        soundfile=sf.__version__, cuda=str(torch.version.cuda))
BEST_NAME = 'qbh_final_v5_best.pth'
LAST_NAME = 'qbh_final_v5_last.pth'
TRAIN_OUTPUT = OUTPUT_ROOT / 'train'
EVAL_OUTPUT = OUTPUT_ROOT / 'test'
TRAIN_OUTPUT.mkdir(exist_ok=True)
EVAL_OUTPUT.mkdir(exist_ok=True)



In [ ]:
# CoverHunter backbone is embedded to keep this notebook self-contained.
import logging
import math
from typing import Union
from typeguard import check_argument_types

class BasicModel(torch.nn.Module):
  """Minimal CoverHunter base class required by the embedded inference model."""

  def model_size(self) -> int:
    return sum(parameter.numel() for parameter in self.parameters())



class Linear(torch.nn.Module):
  def __init__(self, in_dim, out_dim, bias=True, w_init_gain='linear'):
    super(Linear, self).__init__()
    self.linear_layer = torch.nn.Linear(in_dim, out_dim, bias=bias)

    torch.nn.init.xavier_uniform_(
      self.linear_layer.weight,
      gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, x):
    return self.linear_layer(x)


class Conv1d(torch.nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
               padding=None, dilation=1, bias=True, w_init_gain='linear'):
    super(Conv1d, self).__init__()
    if padding is None:
      assert (kernel_size % 2 == 1)
      padding = int(dilation * (kernel_size - 1) / 2)

    self.conv = torch.nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size, stride=stride,
                                padding=padding, dilation=dilation,
                                bias=bias)

    torch.nn.init.xavier_uniform_(
      self.conv.weight, gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, signal):
    conv_signal = self.conv(signal)
    return conv_signal


class CenterLoss(nn.Module):
  """Center loss.

  Reference:
    A Discriminative Feature Learning Approach for Deep Face Recognition.

  Args:
      num_classes (int): number of classes.
      feat_dim (int): feature dimension.
  """

  def __init__(self, num_classes=10, feat_dim=2, use_gpu=True):
    super(CenterLoss, self).__init__()
    self.num_classes = num_classes
    self.feat_dim = feat_dim
    self.use_gpu = use_gpu
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    if self.use_gpu:
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).cuda())
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).to("cuda"))
    else:
      self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))
    logging.info("CenterLoss: num_classes({}), feat_dim({})".format(
      num_classes, feat_dim))
    return

  def forward(self, x, labels):
    """
    Args:
        x: feature matrix with shape (batch_size, feat_dim).
        labels: ground truth labels with shape (batch_size).
    """
    batch_size = x.size(0)
    distmat = torch.pow(x, 2).sum(dim=1, keepdim=True). \
                expand(batch_size, self.num_classes) + \
              torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(
                self.num_classes, batch_size).t()
    distmat.addmm_(1, -2, x, self.centers.t())

    classes = torch.arange(self.num_classes).long()
    if self.use_gpu:
      classes = classes.cuda()
    labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
    mask = labels.eq(classes.expand(batch_size, self.num_classes))

    dist = distmat * mask.float()
    loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size

    return loss


class FocalLoss(nn.Module):
  """ Focal Loss implement for https://arxiv.org/abs/1708.02002 """

  def __init__(self, gamma: float = 2., alpha: Optional[List] = None, num_cls: int = -1,
               reduction: str = 'mean'):

    super(FocalLoss, self).__init__()
    assert check_argument_types()
    if reduction not in ['mean', 'sum']:
      raise NotImplementedError(
        'Reduction {} not implemented.'.format(reduction))
    self._reduction = reduction
    self._alpha = alpha
    self._gamma = gamma
    if alpha is not None:
      assert len(alpha) <= num_cls, "{} != {}".format(len(alpha), num_cls)
      self._alpha = torch.tensor(self._alpha)
    self._eps = np.finfo(float).eps
    logging.info("Init Focal loss with gamma:{}".format(gamma))
    return

  def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """ compute focal loss for pred and label

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss
    """
    b = y_pred.size(0)
    y_pred_softmax = torch.nn.Softmax(dim=1)(y_pred) + self._eps
    ce = -torch.log(y_pred_softmax)
    ce = ce.gather(1, y_true.view(-1, 1))

    y_pred_softmax = y_pred_softmax.gather(1, y_true.view(-1, 1))
    weight = torch.pow(torch.sub(1., y_pred_softmax), self._gamma)

    if self._alpha is not None:
      self._alpha = self._alpha.to(y_pred.device)
      alpha = self._alpha.gather(0, y_true.view(-1))
      alpha = alpha.unsqueeze(1)
      alpha = alpha / torch.sum(alpha) * b
      weight = torch.mul(alpha, weight)
    fl_loss = torch.mul(weight, ce).squeeze(1)
    return self._reduce(fl_loss)

  def forward_with_onehot(self, y_pred: torch.Tensor,
                          y_true: torch.Tensor) -> torch.Tensor:
    """ Another implement for "forward" with onehot label matrix.

    It is not good because when ce_embed size is large, more memory will used.

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss

    """
    y_pred = torch.nn.Softmax(dim=1)(y_pred)
    y_true = torch.nn.functional.one_hot(y_true, self.num_cls)

    eps = np.finfo(float).eps
    y_pred = y_pred + eps
    ce = torch.mul(y_true, -torch.log(y_pred))
    weight = torch.mul(y_true, torch.pow(torch.sub(1., y_pred), self._gamma))
    fl_loss = torch.mul(self.alpha, torch.mul(weight, ce))

    fl_loss, _ = torch.max(fl_loss, dim=1)
    return self._reduce(fl_loss)

  def _reduce(self, x):
    if self._reduction == 'mean':
      return torch.mean(x)
    else:
      return torch.sum(x)


class HardTripletLoss(nn.Module):
  """Hard/Hardest Triplet Loss
    (pytorch implementation of https://omoindrot.github.io/triplet-loss)
    For each anchor, we get the hardest positive and hardest negative to form a triplet.
    """

  def __init__(self, margin=0.1):
    """ Args:
      margin: margin for triplet loss
    """
    super(HardTripletLoss, self).__init__()
    self._margin = margin
    return

  def forward(self, embeddings, labels):
    """
    Args:
        labels: labels of the batch, of size (batch_size,)
        embeddings: tensor of shape (batch_size, embed_dim)

    Returns:
        triplet_loss: scalar tensor containing the triplet loss
    """
    pairwise_dist = self._pairwise_distance(embeddings, squared=False)

    mask_anchor_positive = self._get_anchor_positive_triplet_mask(
      labels).float()
    valid_positive_dist = pairwise_dist * mask_anchor_positive
    hardest_positive_dist, _ = torch.max(valid_positive_dist, dim=1,
                                         keepdim=True)

    # Get the hardest negative pairs
    mask_negative = self._get_anchor_negative_triplet_mask(labels).float()
    max_negative_dist, _ = torch.max(pairwise_dist, dim=1, keepdim=True)
    negative_dist = pairwise_dist + max_negative_dist * (1.0 - mask_negative)
    hardest_negative_dist, _ = torch.min(negative_dist, dim=1, keepdim=True)

    # Combine biggest d(a, p) and smallest d(a, n) into final triplet loss
    triplet_loss = F.relu(
      hardest_positive_dist - hardest_negative_dist + self._margin)
    triplet_loss = torch.mean(triplet_loss)
    return triplet_loss

  @staticmethod
  def _pairwise_distance(x, squared=False, eps=1e-16):
    # Compute the 2D matrix of distances between all the embeddings.

    cor_mat = torch.matmul(x, x.t())
    norm_mat = cor_mat.diag()
    distances = norm_mat.unsqueeze(1) - 2 * cor_mat + norm_mat.unsqueeze(0)
    distances = F.relu(distances)

    if not squared:
      mask = torch.eq(distances, 0.0).float()
      distances = distances + mask * eps
      distances = torch.sqrt(distances)
      distances = distances * (1.0 - mask)
    return distances

  @staticmethod
  def _get_anchor_positive_triplet_mask(labels):
    """Return a 2D mask where mask[a, p] is True, if a and p are distinct and
       have same label.

    """
    # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = labels.device
    indices_not_equal = torch.eye(labels.shape[0]).to(device).byte() ^ 1
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = indices_not_equal * labels_equal
    return mask

  @staticmethod
  def _get_anchor_negative_triplet_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    """
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = labels_equal ^ 1
    return mask


def subsequent_chunk_mask(
    size: int,
    chunk_size: int,
    num_left_chunks: int = -1,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
  """Create mask for subsequent steps (size, size) with chunk size,
       this is for streaming encoder

    Args:
        size (int): size of mask
        chunk_size (int): size of chunk
        num_left_chunks (int): number of left chunks
            <0: use full chunk
            >=0: use num_left_chunks
        device (torch.device): "cpu" or "cuda" or torch.Tensor.device

    Returns:
        torch.Tensor: mask

    Examples:
        >>> subsequent_chunk_mask(4, 2)
        [[1, 1, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 1],
         [1, 1, 1, 1]]
    """
  ret = torch.zeros(size, size, device=device, dtype=torch.bool)
  for i in range(size):
    if num_left_chunks < 0:
      start = 0
    else:
      start = max((i // chunk_size - num_left_chunks) * chunk_size, 0)
    ending = min((i // chunk_size + 1) * chunk_size, size)
    ret[i, start:ending] = True
  return ret


def add_optional_chunk_mask(xs: torch.Tensor, masks: torch.Tensor,
                            use_dynamic_chunk: bool,
                            use_dynamic_left_chunk: bool,
                            decoding_chunk_size: int, static_chunk_size: int,
                            num_decoding_left_chunks: int):
  """ Apply optional mask for encoder.

    Args:
        masks:
        xs (torch.Tensor): padded input, (B, L, D), L for max length
        mask (torch.Tensor): mask for xs, (B, 1, L)
        use_dynamic_chunk (bool): whether to use dynamic chunk or not
        use_dynamic_left_chunk (bool): whether to use dynamic left chunk for
            training.
        decoding_chunk_size (int): decoding chunk size for dynamic chunk, it's
            0: default for training, use random dynamic chunk.
            <0: for decoding, use full chunk.
            >0: for decoding, use fixed chunk size as set.
        static_chunk_size (int): chunk size for static chunk training/decoding
            if it's greater than 0, if use_dynamic_chunk is true,
            this parameter will be ignored
        num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
            >=0: use num_decoding_left_chunks
            <0: use all left chunks

    Returns:
        torch.Tensor: chunk mask of the input xs.
    """
  # Whether to use chunk mask or not
  if use_dynamic_chunk:
    max_len = xs.size(1)
    if decoding_chunk_size < 0:
      chunk_size = max_len
      num_left_chunks = -1
    elif decoding_chunk_size > 0:
      chunk_size = decoding_chunk_size
      num_left_chunks = num_decoding_left_chunks
    else:
      # chunk size is either [1, 25] or full context(max_len).
      # Since we use 4 times subsampling and allow up to 1s(100 frames)
      # delay, the maximum frame is 100 / 4 = 25.
      chunk_size = torch.randint(1, max_len, (1,)).item()
      num_left_chunks = -1
      if chunk_size > max_len // 2:
        chunk_size = max_len
      else:
        chunk_size = chunk_size % 25 + 1
        if use_dynamic_left_chunk:
          max_left_chunks = (max_len - 1) // chunk_size
          num_left_chunks = torch.randint(0, max_left_chunks,
                                          (1,)).item()
    chunk_masks = subsequent_chunk_mask(xs.size(1), chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  elif static_chunk_size > 0:
    num_left_chunks = num_decoding_left_chunks
    chunk_masks = subsequent_chunk_mask(xs.size(1), static_chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  else:
    chunk_masks = masks
  return chunk_masks


def make_pad_mask(lengths: torch.Tensor, max_len: int = 0) -> torch.Tensor:
  """Make mask tensor containing indices of padded part.

    See description of make_non_pad_mask.

    Args:
        lengths (torch.Tensor): Batch of lengths (B,).
        max_len:

    Returns:
        torch.Tensor: Mask tensor containing indices of padded part.

    Examples:
        >>> make_pad_mask(torch.tensor([5, 3, 2]))
        masks = [[0, 0, 0, 0 ,0],
                 [0, 0, 0, 1, 1],
                 [0, 0, 1, 1, 1]]
    """
  batch_size = lengths.size(0)
  max_len = max_len if max_len > 0 else lengths.max().item()
  seq_range = torch.arange(0,
                           max_len,
                           dtype=torch.int64,
                           device=lengths.device)
  seq_range_expand = seq_range.unsqueeze(0).expand(batch_size, max_len)
  seq_length_expand = lengths.unsqueeze(-1)
  mask = seq_range_expand >= seq_length_expand
  return mask


class Swish(torch.nn.Module):
  """Construct an Swish object."""

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """Return Swish activation function."""
    return x * torch.sigmoid(x)


def get_activation(act):
  """Return activation function."""
  activation_funcs = {
    "hardtanh": torch.nn.Hardtanh,
    "tanh": torch.nn.Tanh,
    "relu": torch.nn.ReLU,
    "selu": torch.nn.SELU,
    "swish": getattr(torch.nn, "SiLU", Swish),
    "gelu": torch.nn.GELU
  }
  return activation_funcs[act]()


class PositionalEncoding(torch.nn.Module):
  """Positional encoding.

    :param int d_model: embedding dim
    :param float dropout_rate: dropout rate
    :param int max_len: maximum input length

    PE(pos, 2i)   = sin(pos/(10000^(2i/dmodel)))
    PE(pos, 2i+1) = cos(pos/(10000^(2i/dmodel)))
    """

  def __init__(self,
               d_model: int,
               dropout_rate: float,
               max_len: int = 5000,
               reverse: bool = False):
    """Construct an PositionalEncoding object."""
    super().__init__()
    self.d_model = d_model
    self.xscale = math.sqrt(self.d_model)
    self.dropout = torch.nn.Dropout(p=dropout_rate)
    self.max_len = max_len

    self.pe = torch.zeros(self.max_len, self.d_model)
    position = torch.arange(0, self.max_len,
                            dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
      torch.arange(0, self.d_model, 2, dtype=torch.float32) *
      -(math.log(10000.0) / self.d_model))
    self.pe[:, 0::2] = torch.sin(position * div_term)
    self.pe[:, 1::2] = torch.cos(position * div_term)
    self.pe = self.pe.unsqueeze(0)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Add positional encoding.

        Args:
            x (torch.Tensor): Input. Its shape is (batch, time, ...)
            offset (int, torch.tensor): position offset

        Returns:
            torch.Tensor: Encoded tensor. Its shape is (batch, time, ...)
            torch.Tensor: for compatibility to RelPositionalEncoding
        """

    self.pe = self.pe.to(x.device)
    pos_emb = self.position_encoding(offset, x.size(1), False)
    x = x * self.xscale + pos_emb
    return self.dropout(x), self.dropout(pos_emb)

  def position_encoding(self, offset: Union[int, torch.Tensor], size: int,
                        apply_dropout: bool = True) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

        Attention!!!!!
        we apply dropout only once at the whole utterance level in a none
        streaming way, but will call this function several times with
        increasing input size in a streaming scenario, so the dropout will
        be applied several times.

        Args:
            apply_dropout:
            offset (int or torch.tensor): start offset
            size (int): required size of position encoding

        Returns:
            torch.Tensor: Corresponding encoding
        """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    elif isinstance(offset, torch.Tensor) and offset.dim() == 0:  # scalar
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    else:  # for batched streaming decoding on GPU
      assert torch.max(offset) + size < self.max_len
      index = offset.unsqueeze(1) + torch.arange(0, size).to(offset.device)  # B X T
      flag = index > 0
      # remove negative offset
      index = index * flag
      pos_emb = F.embedding(index, self.pe[0])  # B X T X d_model

    if apply_dropout:
      pos_emb = self.dropout(pos_emb)
    return pos_emb


class RelPositionalEncoding(PositionalEncoding):
  """Relative positional encoding module.
    See : Appendix B in https://arxiv.org/abs/1901.02860
    Args:
        d_model (int): Embedding dimension.
        dropout_rate (float): Dropout rate.
        max_len (int): Maximum input length.
    """

  def __init__(self, d_model: int, dropout_rate: float, max_len: int = 5000):
    """Initialize class."""
    super().__init__(d_model, dropout_rate, max_len, reverse=True)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute positional encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, `*`).
        Returns:
            torch.Tensor: Encoded tensor (batch, time, `*`).
            torch.Tensor: Positional embedding tensor (1, time, `*`).
        """
    self.pe = self.pe.to(x.device)
    x = x * self.xscale
    pos_emb = self.position_encoding(offset, x.size(1), False)
    return self.dropout(x), self.dropout(pos_emb)


class NoPositionalEncoding(torch.nn.Module):
  """ No position encoding
    """

  def __init__(self, d_model: int, dropout_rate: float):
    super().__init__()
    self.d_model = d_model
    self.dropout = torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """ Just return zero vector for interface compatibility
        """
    pos_emb = torch.zeros(1, x.size(1), self.d_model).to(x.device)
    return self.dropout(x), pos_emb

  def position_encoding(
      self, offset: Union[int, torch.Tensor], size: int) -> torch.Tensor:
    return torch.zeros(1, size, self.d_model)


class BaseSubsampling(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.right_context = 0
    self.subsampling_rate = 1

  def position_encoding(self, offset: Union[int, torch.Tensor],
                        size: int) -> torch.Tensor:
    return self.pos_enc.position_encoding(offset, size)


class LinearNoSubsampling(BaseSubsampling):
  """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an linear object."""
    super().__init__()
    self.out = torch.nn.Sequential(
      torch.nn.Linear(idim, odim),
      torch.nn.LayerNorm(odim, eps=1e-5),
      torch.nn.Dropout(dropout_rate),
    )
    self.pos_enc = pos_enc_class
    self.right_context = 0
    self.subsampling_rate = 1

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
    x = self.out(x)
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask


class Conv2dSubsampling4(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/4 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling4 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.out = torch.nn.Sequential(
      torch.nn.Linear(odim * (((idim - 1) // 2 - 1) // 2), odim))
    self.pos_enc = pos_enc_class
    # The right context for every conv layer is computed by:
    # (kernel_size - 1) * frame_rate_of_this_layer
    self.subsampling_rate = 4
    # 6 = (3 - 1) * 1 + (3 - 1) * 2
    self.right_context = 6

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 4.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 4.
            torch.Tensor: positional encoding

        """
    x = x.unsqueeze(1)  # (b, c=1, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.out(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2]


class Conv2dSubsampling8(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/8 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling8 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.linear = torch.nn.Linear(
      odim * ((((idim - 1) // 2 - 1) // 2 - 1) // 2), odim)
    self.pos_enc = pos_enc_class
    self.subsampling_rate = 8
    # 14 = (3 - 1) * 1 + (3 - 1) * 2 + (3 - 1) * 4
    self.right_context = 14

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 8.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 8.
            torch.Tensor: positional encoding
        """
    x = x.unsqueeze(1)  # (b, c, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2][:, :, :-2:2]


class MultiHeadedAttention(nn.Module):
  """Multi-Head Attention layer.

    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, n_head: int, n_feat: int, dropout_rate: float):
    """Construct an MultiHeadedAttention object."""
    super().__init__()
    assert n_feat % n_head == 0
    # We assume d_v always equals d_k
    self.d_k = n_feat // n_head
    self.h = n_head
    self.linear_q = nn.Linear(n_feat, n_feat)
    self.linear_k = nn.Linear(n_feat, n_feat)
    self.linear_v = nn.Linear(n_feat, n_feat)
    self.linear_out = nn.Linear(n_feat, n_feat)
    self.dropout = nn.Dropout(p=dropout_rate)

  def forward_qkv(
      self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transform query, key and value.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).

        Returns:
            torch.Tensor: Transformed query tensor, size
                (#batch, n_head, time1, d_k).
            torch.Tensor: Transformed key tensor, size
                (#batch, n_head, time2, d_k).
            torch.Tensor: Transformed value tensor, size
                (#batch, n_head, time2, d_k).

        """
    n_batch = query.size(0)
    q = self.linear_q(query).view(n_batch, -1, self.h, self.d_k)
    k = self.linear_k(key).view(n_batch, -1, self.h, self.d_k)
    v = self.linear_v(value).view(n_batch, -1, self.h, self.d_k)
    q = q.transpose(1, 2)  # (batch, head, time1, d_k)
    k = k.transpose(1, 2)  # (batch, head, time2, d_k)
    v = v.transpose(1, 2)  # (batch, head, time2, d_k)

    return q, k, v

  def forward_attention(
      self, value: torch.Tensor, scores: torch.Tensor,
      mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool)
  ) -> torch.Tensor:
    """Compute attention context vector.

        Args:
            value (torch.Tensor): Transformed value, size
                (#batch, n_head, time2, d_k).
            scores (torch.Tensor): Attention score, size
                (#batch, n_head, time1, time2).
            mask (torch.Tensor): Mask, size (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.

        Returns:
            torch.Tensor: Transformed value (#batch, time1, d_model)
                weighted by the attention score (#batch, time1, time2).

        """
    n_batch = value.size(0)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be True?
    #   1. onnx(16/4) [WHY? Because we feed real cache & real mask for the
    #           1st chunk to ease the onnx export.]
    #   2. pytorch training
    if mask.size(2) > 0:  # time2 > 0
      mask = mask.unsqueeze(1).eq(0)  # (batch, 1, *, time2)
      # For last chunk, time2 might be larger than scores.size(-1)
      mask = mask[:, :, :, :scores.size(-1)]  # (batch, 1, *, time2)
      scores = scores.masked_fill(mask, -float('inf'))
      attn = torch.softmax(scores, dim=-1).masked_fill(
        mask, 0.0)  # (batch, head, time1, time2)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be False?
    #   1. onnx(16/-1, -1/-1, 16/0)
    #   2. jit (16/-1, -1/-1, 16/0, 16/4)
    else:
      attn = torch.softmax(scores, dim=-1)  # (batch, head, time1, time2)

    p_attn = self.dropout(attn)
    x = torch.matmul(p_attn, value)  # (batch, head, time1, d_k)
    x = (x.transpose(1, 2).contiguous().view(n_batch, -1,
                                             self.h * self.d_k)
         )  # (batch, time1, d_model)

    return self.linear_out(x)  # (batch, time1, d_model)

  def forward(self, query: torch.Tensor, key: torch.Tensor,
              value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot product attention.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2).
                1.When applying cross attention between decoder and encoder,
                the batch padding mask for input is in (#batch, 1, T) shape.
                2.When applying self attention of encoder,
                the mask is in (#batch, T, T)  shape.
                3.When applying self attention of decoder,
                the mask is in (#batch, L, L)  shape.
                4.If the different position in decoder see different block
                of the encoder, such as Mocha, the passed in mask could be
                in (#batch, L, T) shape. But there is no such case in current
                Wenet.
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`


        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`

        """
    q, k, v = self.forward_qkv(query, key, value)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
    return self.forward_attention(v, scores, mask), new_cache


class RelPositionMultiHeadedAttention(MultiHeadedAttention):
  """Multi-Head Attention layer with relative position encoding.
    Paper: https://arxiv.org/abs/1901.02860
    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.
    """

  def __init__(self, n_head, n_feat, dropout_rate):
    """Construct an RelPositionMultiHeadedAttention object."""
    super().__init__(n_head, n_feat, dropout_rate)
    # linear transformation for positional encoding
    self.linear_pos = nn.Linear(n_feat, n_feat, bias=False)
    # these two learnable bias are used in matrix c and matrix d
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    self.pos_bias_u = nn.Parameter(torch.Tensor(self.h, self.d_k))
    self.pos_bias_v = nn.Parameter(torch.Tensor(self.h, self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self, x, zero_triu: bool = False):
    """Compute relative positinal encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, size).
            zero_triu (bool): If true, return the lower triangular part of
                the matrix.
        Returns:
            torch.Tensor: Output tensor.
        """

    zero_pad = torch.zeros((x.size()[0], x.size()[1], x.size()[2], 1),
                           device=x.device,
                           dtype=x.dtype)
    x_padded = torch.cat([zero_pad, x], dim=-1)

    x_padded = x_padded.view(x.size()[0],
                             x.size()[1],
                             x.size(3) + 1, x.size(2))
    x = x_padded[:, :, 1:].view_as(x)

    if zero_triu:
      ones = torch.ones((x.size(2), x.size(3)))
      x = x * torch.tril(ones, x.size(3) - x.size(2))[None, None, :, :]

    return x

  def forward(self, query: torch.Tensor,
              key: torch.Tensor, value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute 'Scaled Dot Product Attention' with rel. positional encoding.
        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): Positional embedding tensor
                (#batch, time2, size).
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        """
    q, k, v = self.forward_qkv(query, key, value)
    q = q.transpose(1, 2)  # (batch, time1, head, d_k)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    n_batch_pos = pos_emb.size(0)
    p = self.linear_pos(pos_emb).view(n_batch_pos, -1, self.h, self.d_k)
    p = p.transpose(1, 2)  # (batch, head, time1, d_k)

    # (batch, head, time1, d_k)
    q_with_bias_u = (q + self.pos_bias_u).transpose(1, 2)
    # (batch, head, time1, d_k)
    q_with_bias_v = (q + self.pos_bias_v).transpose(1, 2)

    # compute attention score
    # first compute matrix a and matrix c
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    # (batch, head, time1, time2)
    matrix_ac = torch.matmul(q_with_bias_u, k.transpose(-2, -1))

    # compute matrix b and matrix d
    # (batch, head, time1, time2)
    matrix_bd = torch.matmul(q_with_bias_v, p.transpose(-2, -1))
    # Remove rel_shift since it is useless in speech recognition,
    # and it requires special attention for streaming.
    # matrix_bd = self.rel_shift(matrix_bd)

    scores = (matrix_ac + matrix_bd) / math.sqrt(
      self.d_k)  # (batch, head, time1, time2)

    return self.forward_attention(v, scores, mask), new_cache


class ConvolutionModule(nn.Module):
  """ConvolutionModule in Conformer model."""

  def __init__(self,
               channels: int,
               kernel_size: int = 15,
               activation: nn.Module = nn.ReLU(),
               norm: str = "batch_norm",
               causal: bool = False,
               bias: bool = True):
    """Construct an ConvolutionModule object.
        Args:
            channels (int): The number of channels of conv layers.
            kernel_size (int): Kernel size of conv layers.
            causal (int): Whether use causal convolution or not
        """
    assert check_argument_types()
    super().__init__()

    self.pointwise_conv1 = nn.Conv1d(
      channels,
      2 * channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    # self.lorder is used to distinguish if it's a causal convolution,
    # if self.lorder > 0: it's a causal convolution, the input will be
    #    padded with self.lorder frames on the left in forward.
    # else: it's a symmetrical convolution
    if causal:
      padding = 0
      self.lorder = kernel_size - 1
    else:
      # kernel_size should be an odd number for none causal convolution
      assert (kernel_size - 1) % 2 == 0
      padding = (kernel_size - 1) // 2
      self.lorder = 0
    self.depthwise_conv = nn.Conv1d(
      channels,
      channels,
      kernel_size,
      stride=1,
      padding=padding,
      groups=channels,
      bias=bias,
    )

    assert norm in ['batch_norm', 'layer_norm']
    if norm == "batch_norm":
      self.use_layer_norm = False
      self.norm = nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm = True
      self.norm = nn.LayerNorm(channels)

    self.pointwise_conv2 = nn.Conv1d(
      channels,
      channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    self.activation = activation

  def forward(
      self,
      x: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      cache: torch.Tensor = torch.zeros((0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute convolution module.
        Args:
            x (torch.Tensor): Input tensor (#batch, time, channels).
            mask_pad (torch.Tensor): used for batch padding (#batch, 1, time),
                (0, 0, 0) means fake mask.
            cache (torch.Tensor): left context cache, it is only
                used in causal convolution (#batch, channels, cache_t),
                (0, 0, 0) meas fake cache.
        Returns:
            torch.Tensor: Output tensor (#batch, time, channels).
        """
    # exchange the temporal dimension and the feature dimension
    x = x.transpose(1, 2)  # (#batch, channels, time)

    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    if self.lorder > 0:
      if cache.size(2) == 0:  # cache_t == 0
        x = nn.functional.pad(x, (self.lorder, 0), 'constant', 0.0)
      else:
        assert cache.size(0) == x.size(0)  # equal batch
        assert cache.size(1) == x.size(1)  # equal channel
        x = torch.cat((cache, x), dim=2)
      assert (x.size(2) > self.lorder)
      new_cache = x[:, :, -self.lorder:]
    else:
      # It's better we just return None if no cache is required,
      # However, for JIT export, here we just fake one tensor instead of
      # None.
      new_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)

    # GLU mechanism
    x = self.pointwise_conv1(x)  # (batch, 2*channel, dim)
    x = nn.functional.glu(x, dim=1)  # (batch, channel, dim)

    # 1D Depthwise Conv
    x = self.depthwise_conv(x)
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.activation(self.norm(x))
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.pointwise_conv2(x)
    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    return x.transpose(1, 2), new_cache


class PositionwiseFeedForward(torch.nn.Module):
  """Positionwise feed forward layer.

    FeedForward are appied on each position of the sequence.
    The output dim is same with the input dim.

    Args:
        idim (int): Input dimenstion.
        hidden_units (int): The number of hidden units.
        dropout_rate (float): Dropout rate.
        activation (torch.nn.Module): Activation function
    """

  def __init__(self,
               idim: int,
               hidden_units: int,
               dropout_rate: float,
               activation: torch.nn.Module = torch.nn.ReLU()):
    """Construct a PositionwiseFeedForward object."""
    super(PositionwiseFeedForward, self).__init__()
    self.w_1 = torch.nn.Linear(idim, hidden_units)
    self.activation = activation
    self.dropout = torch.nn.Dropout(dropout_rate)
    self.w_2 = torch.nn.Linear(hidden_units, idim)

  def forward(self, xs: torch.Tensor) -> torch.Tensor:
    """Forward function.

        Args:
            xs: input tensor (B, L, D)
        Returns:
            output tensor, (B, L, D)
        """
    return self.w_2(self.dropout(self.activation(self.w_1(xs))))


class ConformerEncoderLayer(torch.nn.Module):
  """Encoder layer module.
    Args:
        size (int): Input dimension.
        self_attn (torch.nn.Module): Self-attention module instance.
            `MultiHeadedAttention` or `RelPositionMultiHeadedAttention`
            instance can be used as the argument.
        feed_forward (torch.nn.Module): Feed-forward module instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        feed_forward_macaron (torch.nn.Module): Additional feed-forward module
             instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        conv_module (torch.nn.Module): Convolution module instance.
            `ConvlutionModule` instance can be used as the argument.
        dropout_rate (float): Dropout rate.
        normalize_before (bool):
            True: use layer_norm before each sub-block.
            False: use layer_norm after each sub-block.
        concat_after (bool): Whether to concat attention layer's input and
            output.
            True: x -> x + linear(concat(x, att(x)))
            False: x -> x + att(x)
    """

  def __init__(
      self,
      size: int,
      self_attn: torch.nn.Module,
      feed_forward: Optional[nn.Module] = None,
      feed_forward_macaron: Optional[nn.Module] = None,
      conv_module: Optional[nn.Module] = None,
      dropout_rate: float = 0.1,
      normalize_before: bool = True,
      concat_after: bool = False,
  ):
    """Construct an EncoderLayer object."""
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.feed_forward_macaron = feed_forward_macaron
    self.conv_module = conv_module
    self.norm_ff = nn.LayerNorm(size, eps=1e-5)  # for the FNN module
    self.norm_mha = nn.LayerNorm(size, eps=1e-5)  # for the MHA module
    if feed_forward_macaron is not None:
      self.norm_ff_macaron = nn.LayerNorm(size, eps=1e-5)
      self.ff_scale = 0.5
    else:
      self.ff_scale = 1.0
    if self.conv_module is not None:
      self.norm_conv = nn.LayerNorm(size,
                                    eps=1e-5)  # for the CNN module
      self.norm_final = nn.LayerNorm(
        size, eps=1e-5)  # for the final output of the block
    self.dropout = nn.Dropout(dropout_rate)
    self.size = size
    self.normalize_before = normalize_before
    self.concat_after = concat_after
    if self.concat_after:
      self.concat_linear = nn.Linear(size + size, size)
    else:
      self.concat_linear = nn.Identity()

  def forward(
      self,
      x: torch.Tensor,
      mask: torch.Tensor,
      pos_emb: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
      cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute encoded features.

        Args:
            x (torch.Tensor): (#batch, time, size)
            mask (torch.Tensor): Mask tensor for the input (#batch, time，time),
                (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): positional encoding, must not be None
                for ConformerEncoderLayer.
            mask_pad (torch.Tensor): batch padding mask used for conv module.
                (#batch, 1，time), (0, 0, 0) means fake mask.
            att_cache (torch.Tensor): Cache tensor of the KEY & VALUE
                (#batch=1, head, cache_t1, d_k * 2), head * d_k == size.
            cnn_cache (torch.Tensor): Convolution cache in conformer layer
                (#batch=1, size, cache_t2)
        Returns:
            torch.Tensor: Output tensor (#batch, time, size).
            torch.Tensor: Mask tensor (#batch, time, time).
            torch.Tensor: att_cache tensor,
                (#batch=1, head, cache_t1 + time, d_k * 2).
            torch.Tensor: cnn_cahce tensor (#batch, size, cache_t2).
        """

    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)

    x_att, new_att_cache = self.self_attn(
      x, x, x, mask, pos_emb, att_cache)
    if self.concat_after:
      x_concat = torch.cat((x, x_att), dim=-1)
      x = residual + self.concat_linear(x_concat)
    else:
      x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache


class ConformerEncoder(torch.nn.Module):
  def __init__(
      self,
      input_size: int,
      output_size: int,
      attention_heads: int = 4,
      linear_units: int = 2048,
      num_blocks: int = 6,
      dropout_rate: float = 0.1,
      positional_dropout_rate: float = 0.1,
      attention_dropout_rate: float = 0.0,
      input_layer: str = "conv2d",
      pos_enc_layer_type: str = "rel_pos",
      normalize_before: bool = True,
      concat_after: bool = False,
      static_chunk_size: int = 0,
      use_dynamic_chunk: bool = False,
      global_cmvn: Optional[torch.nn.Module] = None,
      use_dynamic_left_chunk: bool = False,
      positionwise_conv_kernel_size: int = 1,
      macaron_style: bool = True,
      selfattention_layer_type: str = "rel_selfattn",
      activation_type: str = "swish",
      use_cnn_module: bool = True,
      cnn_module_kernel: int = 15,
      causal: bool = False,
      cnn_module_norm: str = "batch_norm",
  ):
    """
        Args:
            input_size (int): input dim
            output_size (int): dimension of attention
            attention_heads (int): the number of heads of multi head attention
            linear_units (int): the hidden units number of position-wise feed
                forward
            num_blocks (int): the number of decoder blocks
            dropout_rate (float): dropout rate
            attention_dropout_rate (float): dropout rate in attention
            positional_dropout_rate (float): dropout rate after adding
                positional encoding
            input_layer (str): input layer type.
                optional [linear, conv2d, conv2d6, conv2d8]
            pos_enc_layer_type (str): Encoder positional encoding layer type.
                opitonal [abs_pos, scaled_abs_pos, rel_pos, no_pos]
            normalize_before (bool):
                True: use layer_norm before each sub-block of a layer.
                False: use layer_norm after each sub-block of a layer.
            concat_after (bool): whether to concat attention layer's input
                and output.
                True: x -> x + linear(concat(x, att(x)))
                False: x -> x + att(x)
            static_chunk_size (int): chunk size for static chunk training and
                decoding
            use_dynamic_chunk (bool): whether use dynamic chunk size for
                training or not, You can only use fixed chunk(chunk_size > 0)
                or dyanmic chunk size(use_dynamic_chunk = True)
            global_cmvn (Optional[torch.nn.Module]): Optional GlobalCMVN module
            use_dynamic_left_chunk (bool): whether use dynamic left chunk in
                dynamic chunk training
        """
    assert check_argument_types()
    logging.info("Init Conformer layers::")
    super().__init__()
    self._output_size = output_size

    if pos_enc_layer_type == "abs_pos":
      pos_enc_class = PositionalEncoding
    elif pos_enc_layer_type == "rel_pos":
      pos_enc_class = RelPositionalEncoding
    elif pos_enc_layer_type == "no_pos":
      pos_enc_class = NoPositionalEncoding
    else:
      raise ValueError("unknown pos_enc_layer: " + pos_enc_layer_type)
    logging.info("Pos layers - {}".format(pos_enc_layer_type))

    if input_layer == "linear":
      subsampling_class = LinearNoSubsampling
    elif input_layer == "conv2d":
      subsampling_class = Conv2dSubsampling4
    elif input_layer == "conv2d8":
      subsampling_class = Conv2dSubsampling8
    else:
      raise ValueError("unknown input_layer: " + input_layer)
    logging.info("Input layers - {}".format(input_layer))

    self.global_cmvn = global_cmvn
    self.embed = subsampling_class(
      input_size,
      output_size,
      dropout_rate,
      pos_enc_class(output_size, positional_dropout_rate),
    )

    self.normalize_before = normalize_before
    self.after_norm = torch.nn.LayerNorm(output_size, eps=1e-5)
    self.static_chunk_size = static_chunk_size
    self.use_dynamic_chunk = use_dynamic_chunk
    self.use_dynamic_left_chunk = use_dynamic_left_chunk

    activation = get_activation(activation_type)

    # self-attention module definition
    if pos_enc_layer_type != "rel_pos":
      encoder_selfattn_layer = MultiHeadedAttention
    else:
      encoder_selfattn_layer = RelPositionMultiHeadedAttention
    encoder_selfattn_layer_args = (
      attention_heads,
      output_size,
      attention_dropout_rate,
    )
    logging.info("self attn args: {}".format(encoder_selfattn_layer_args))

    # feed-forward module definition
    positionwise_layer = PositionwiseFeedForward
    positionwise_layer_args = (
      output_size,
      linear_units,
      dropout_rate,
      activation,
    )
    logging.info("position-wise args: {}".format(positionwise_layer_args))

    # convolution module definition
    convolution_layer = ConvolutionModule
    convolution_layer_args = (output_size, cnn_module_kernel, activation,
                              cnn_module_norm, causal)

    self.encoders = torch.nn.ModuleList([
      ConformerEncoderLayer(
        output_size,
        encoder_selfattn_layer(*encoder_selfattn_layer_args),
        positionwise_layer(*positionwise_layer_args),
        positionwise_layer(
          *positionwise_layer_args) if macaron_style else None,
        convolution_layer(
          *convolution_layer_args) if use_cnn_module else None,
        dropout_rate,
        normalize_before,
        concat_after,
      ) for _ in range(num_blocks)
    ])
    return

  def output_size(self) -> int:
    return self._output_size

  def forward(
      self,
      xs: torch.Tensor,
      xs_lens: torch.Tensor,
      decoding_chunk_size: int = -1,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        """
    T = xs.size(1)
    masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    xs, pos_emb, masks = self.embed(xs, masks)
    mask_pad = masks  # (B, 1, T/subsample_rate)
    chunk_masks = add_optional_chunk_mask(xs, masks,
                                          self.use_dynamic_chunk,
                                          self.use_dynamic_left_chunk,
                                          decoding_chunk_size,
                                          self.static_chunk_size,
                                          num_decoding_left_chunks)
    for layer in self.encoders:
      xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
    if self.normalize_before:
      xs = self.after_norm(xs)
    # Here we assume the mask is not changed in encoder layers, so just
    # return the masks before encoder layers, and the masks will be used
    # for cross attention with decoder later
    return xs, masks

  def forward_chunk(
      self,
      xs: torch.Tensor,
      offset: int,
      required_cache_size: int,
      att_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      cnn_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      att_mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """ Forward just one chunk

        Args:
            xs (torch.Tensor): chunk input, with shape (b=1, time, mel-dim),
                where `time == (chunk_size - 1) * subsample_rate + \
                        subsample.right_context + 1`
            offset (int): current offset in encoder output time stamp
            required_cache_size (int): cache size required for next chunk
                compuation
                >=0: actual cache size
                <0: means all history cache is required
            att_cache (torch.Tensor): cache tensor for KEY & VALUE in
                transformer/conformer attention, with shape
                (elayers, head, cache_t1, d_k * 2), where
                `head * d_k == hidden-dim` and
                `cache_t1 == chunk_size * num_decoding_left_chunks`.
            cnn_cache (torch.Tensor): cache tensor for cnn_module in conformer,
                (elayers, b=1, hidden-dim, cache_t2), where
                `cache_t2 == cnn.lorder - 1`

        Returns:
            torch.Tensor: output of current input xs,
                with shape (b=1, chunk_size, hidden-dim).
            torch.Tensor: new attention cache required for next chunk, with
                dynamic shape (elayers, head, ?, d_k * 2)
                depending on required_cache_size.
            torch.Tensor: new conformer cnn cache required for next chunk, with
                same shape as the original cnn_cache.

        """
    assert xs.size(0) == 1
    # tmp_masks is just for interface compatibility
    tmp_masks = torch.ones(1,
                           xs.size(1),
                           device=xs.device,
                           dtype=torch.bool)
    tmp_masks = tmp_masks.unsqueeze(1)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    # NOTE(xcsong): Before embed, shape(xs) is (b=1, time, mel-dim)
    xs, pos_emb, _ = self.embed(xs, tmp_masks, offset)
    # NOTE(xcsong): After  embed, shape(xs) is (b=1, chunk_size, hidden-dim)
    elayers, cache_t1 = att_cache.size(0), att_cache.size(2)
    chunk_size = xs.size(1)
    attention_key_size = cache_t1 + chunk_size
    pos_emb = self.embed.position_encoding(
      offset=offset - cache_t1, size=attention_key_size)
    if required_cache_size < 0:
      next_cache_start = 0
    elif required_cache_size == 0:
      next_cache_start = attention_key_size
    else:
      next_cache_start = max(attention_key_size - required_cache_size, 0)
    r_att_cache = []
    r_cnn_cache = []
    for i, layer in enumerate(self.encoders):
      # NOTE(xcsong): Before layer.forward
      #   shape(att_cache[i:i + 1]) is (1, head, cache_t1, d_k * 2),
      #   shape(cnn_cache[i])       is (b=1, hidden-dim, cache_t2)
      xs, _, new_att_cache, new_cnn_cache = layer(
        xs, att_mask, pos_emb,
        att_cache=att_cache[i:i + 1] if elayers > 0 else att_cache,
        cnn_cache=cnn_cache[i] if cnn_cache.size(0) > 0 else cnn_cache
      )
      # NOTE(xcsong): After layer.forward
      #   shape(new_att_cache) is (1, head, attention_key_size, d_k * 2),
      #   shape(new_cnn_cache) is (b=1, hidden-dim, cache_t2)
      r_att_cache.append(new_att_cache[:, :, next_cache_start:, :])
      r_cnn_cache.append(new_cnn_cache.unsqueeze(0))
    if self.normalize_before:
      xs = self.after_norm(xs)

    # NOTE(xcsong): shape(r_att_cache) is (elayers, head, ?, d_k * 2),
    #   ? may be larger than cache_t1, it depends on required_cache_size
    r_att_cache = torch.cat(r_att_cache, dim=0)
    # NOTE(xcsong): shape(r_cnn_cache) is (e, b=1, hidden-dim, cache_t2)
    r_cnn_cache = torch.cat(r_cnn_cache, dim=0)

    return (xs, r_att_cache, r_cnn_cache)

  def forward_chunk_by_chunk(
      self,
      xs: torch.Tensor,
      decoding_chunk_size: int,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """ Forward input chunk by chunk with chunk_size like a streaming
            fashion

        Here we should pay special attention to computation cache in the
        streaming style forward chunk by chunk. Three things should be taken
        into account for computation in the current network:
            1. transformer/conformer encoder layers output cache
            2. convolution in conformer
            3. convolution in subsampling

        However, we don't implement subsampling cache for:
            1. We can control subsampling module to output the right result by
               overlapping input instead of cache left context, even though it
               wastes some computation, but subsampling only takes a very
               small fraction of computation in the whole model.
            2. Typically, there are several covolution layers with subsampling
               in subsampling module, it is tricky and complicated to do cache
               with different convolution layers with different subsampling
               rate.
            3. Currently, nn.Sequential is used to stack all the convolution
               layers in subsampling, we need to rewrite it to make it work
               with cache, which is not prefered.
        Args:
            xs (torch.Tensor): (1, max_len, dim)
            chunk_size (int): decoding chunk size
        """
    assert decoding_chunk_size > 0
    # The model is trained by static or dynamic chunk
    assert self.static_chunk_size > 0 or self.use_dynamic_chunk
    subsampling = self.embed.subsampling_rate
    context = self.embed.right_context + 1  # Add current frame
    stride = subsampling * decoding_chunk_size
    decoding_window = (decoding_chunk_size - 1) * subsampling + context
    num_frames = xs.size(1)
    att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    outputs = []
    offset = 0
    required_cache_size = decoding_chunk_size * num_decoding_left_chunks

    # Feed forward overlap input step by step
    for cur in range(0, num_frames - context + 1, stride):
      end = min(cur + decoding_window, num_frames)
      chunk_xs = xs[:, cur:end, :]
      (y, att_cache, cnn_cache) = self.forward_chunk(
        chunk_xs, offset, required_cache_size, att_cache, cnn_cache)
      outputs.append(y)
      offset += y.size(1)
    ys = torch.cat(outputs, 1)
    masks = torch.ones((1, 1, ys.size(1)), device=ys.device, dtype=torch.bool)
    return ys, masks


class AttentiveStatisticsPooling(torch.nn.Module):
  """This class implements an attentive statistic pooling layer for each channel.
  It returns the concatenated mean and std of the input tensor.

  Arguments
  ---------
  channels: int
      The number of input channels.
  output_channels: int
      The number of output channels.
  """

  def __init__(self, channels, output_channels):
    super().__init__()

    self._eps = 1e-12
    self._linear = Linear(channels * 3, channels)
    self._tanh = torch.nn.Tanh()
    self._conv = Conv1d(
      in_channels=channels, out_channels=channels, kernel_size=1
    )
    self._final_layer = torch.nn.Linear(channels * 2, output_channels,
                                        bias=False)
    logging.info("Init AttentiveStatisticsPooling with {}->{}".format(
      channels, output_channels))
    return

  @staticmethod
  def _compute_statistics(x: torch.Tensor,
                          m: torch.Tensor,
                          eps: float,
                          dim: int = 2):
    mean = (m * x).sum(dim)
    std = torch.sqrt((m * (x - mean.unsqueeze(dim)).pow(2)).sum(dim).clamp(eps))
    return mean, std

  def forward(self, x: torch.Tensor):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, L, C].
    """

    x = x.transpose(1, 2)
    L = x.shape[-1]
    lengths = torch.ones(x.shape[0], device=x.device)
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)
    total = mask.sum(dim=2, keepdim=True).float()

    mean, std = self._compute_statistics(x, mask / total, self._eps)
    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)
    attn = self._conv(self._tanh(self._linear(
      attn.transpose(1, 2)).transpose(1, 2)))

    attn = attn.masked_fill(mask == 0, float("-inf"))  # Filter out zero-padding
    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    pooled_stats = self._final_layer(torch.cat((mean, std), dim=1))
    return pooled_stats

  def forward_with_mask(self, x: torch.Tensor,
                        lengths: Optional[torch.Tensor] = None):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, C, L].
      lengths:
    """
    L = x.shape[-1]

    if lengths is None:
      lengths = torch.ones(x.shape[0], device=x.device)

    # Make binary mask of shape [N, 1, L]
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)

    # Expand the temporal context of the pooling layer by allowing the
    # self-attention to look at global properties of the utterance.

    # torch.std is unstable for backward computation
    # https://github.com/pytorch/pytorch/issues/4320
    total = mask.sum(dim=2, keepdim=True).float()
    mean, std = self._compute_statistics(x, mask / total, self._eps)

    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)

    # Apply layers
    attn = self.conv(self._tanh(self._linear(attn, lengths)))

    # Filter out zero-paddings
    attn = attn.masked_fill(mask == 0, float("-inf"))

    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    # Append mean and std of the batch
    pooled_stats = torch.cat((mean, std), dim=1)
    pooled_stats = pooled_stats.unsqueeze(2)
    return pooled_stats

  @staticmethod
  def length_to_mask(length: torch.Tensor,
                     max_len: Optional[int] = None,
                     dtype: Optional[torch.dtype] = None,
                     device: Optional[torch.device] = None):
    """Creates a binary mask for each sequence.

    Arguments
    ---------
    length : torch.LongTensor
        Containing the length of each sequence in the batch. Must be 1D.
    max_len : int
        Max length for the mask, also the size of the second dimension.
    dtype : torch.dtype, default: None
        The dtype of the generated mask.
    device: torch.device, default: None
        The device to put the mask variable.

    Returns
    -------
    mask : tensor
        The binary mask.

    Example
    -------
    """
    assert len(length.shape) == 1

    if max_len is None:
      max_len = length.max().long().item()  # using arange to generate mask
    mask = torch.arange(
      max_len, device=length.device, dtype=length.dtype
    ).expand(len(length), max_len) < length.unsqueeze(1)

    if dtype is None:
      dtype = length.dtype

    if device is None:
      device = length.device

    mask = torch.as_tensor(mask, dtype=dtype, device=device)
    return mask


class Model(BasicModel):
  """ Csi Backbone Model:
      Batch-norm
      Conformer-Encoder(x6 or x4)
      Global-Avg-Pool
      Resize and linear
      Bn-neck

  Loss:
    Focal-loss(ce) + Triplet-loss + Center-loss

  Args:
    hp: Dict with all parameters
  """

  def __init__(self, hp: Dict):
    super(BasicModel, self).__init__()
    self._hp = hp

    self._global_cmvn = torch.nn.BatchNorm1d(hp["input_dim"])
    self._encoder = ConformerEncoder(
      input_size=hp["input_dim"],
      output_size=hp["encoder"]["output_dims"],
      linear_units=hp["encoder"]["attention_dim"],
      num_blocks=hp["encoder"]["num_blocks"])

    if hp["encoder"]["output_dims"] != hp["embed_dim"]:
      self._embed_lo = torch.nn.Linear(hp["encoder"]["output_dims"],
                                       hp["embed_dim"])
    else:
      self._embed_lo = None

    # Bottleneck
    self._bottleneck = torch.nn.BatchNorm1d(hp["embed_dim"])
    self._bottleneck.bias.requires_grad_(False)  # no shift

    self._pool_layer = AttentiveStatisticsPooling(
      hp["embed_dim"], output_channels=hp["embed_dim"])
    self._ce_layer = torch.nn.Linear(
      hp["embed_dim"], hp["ce"]["output_dims"], bias=False)

    # Loss
    if "alpha" in hp["ce"].keys():
      alpha = np.load(hp["ce"]["alpha"])
      alpha = 1.0 / (alpha + 1)
      alpha = alpha / np.sum(alpha)
      logging.warning("use alpha with {}".format(len(alpha)))
    else:
      alpha = None
      logging.warning("Not use alpha")

    self._ce_loss = FocalLoss(alpha=alpha, gamma=self._hp["ce"]["gamma"],
                              num_cls=self._hp["ce"]["output_dims"])
    self._triplet_loss = HardTripletLoss(margin=hp["triplet"]["margin"])
    self._center_loss = CenterLoss(num_classes=self._hp["ce"]["output_dims"],
                                   feat_dim=hp["embed_dim"], use_gpu=torch.cuda.is_available())
    logging.info(
      "Model size: {:.3f}M \n".format(self.model_size() / 1000 / 1000))
    return

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """feat[b, frame_size, feat_size] -> embed[b, embed_dim]"""

    x = self._global_cmvn(x.transpose(1, 2)).transpose(1, 2)
    xs_lens = torch.full(
      [x.size(0)], fill_value=x.size(1), dtype=torch.long).to(x.device).long()
    x, _ = self._encoder(x, xs_lens=xs_lens, decoding_chunk_size=-1)
    x = self._pool_layer(x)
    return x

  @torch.jit.ignore
  def compute_loss(self, anchor: torch.Tensor, label: torch.Tensor) -> Tuple[
    torch.Tensor, Dict]:
    """compute ce and triplet loss"""
    f_t = self.forward(anchor)
    # print("f_t", f_t)

    f_i = self._bottleneck(f_t)
    ce_pred = self._ce_layer(f_i)
    # print("ce_pred", ce_pred)
    ce_loss = self._ce_loss(ce_pred, label)
    loss = ce_loss * self._hp["ce"]["weight"]
    loss_dict = {"ce_loss": ce_loss}

    tri_weight = self._hp["triplet"]["weight"]
    if tri_weight > 0.01:
      tri_loss = self._triplet_loss(f_t, label)
      loss = loss + tri_loss * tri_weight
      loss_dict.update({"tri_loss": tri_loss})

    cen_weight = self._hp["center"]["weight"]
    if cen_weight > 0.0:
      cen_loss = self._center_loss(f_t, label)
      loss = loss + cen_loss * cen_weight
      loss_dict.update({"cen_loss": cen_loss})
    return loss, loss_dict

  @torch.jit.ignore
  def inference(self, feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
      embed = self.forward(feat)
      embed_ce = self._ce_layer(embed)
    return embed, embed_ce

  @torch.jit.export
  def compute_embed(self, feat: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
      embed = self.forward(feat)
    return embed


In [ ]:
class AudioPreprocessor:
    """Tiền xử lý âm thanh: Resample về 16,000 Hz, chuẩn hóa biên độ và cắt Sliding Window 10s / hop 3s."""
    def __init__(self, sample_rate: int = 16000, chunk_duration: float = 10.0, hop_duration: float = 3.0):
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.hop_duration = hop_duration
        self.chunk_samples = int(chunk_duration * sample_rate)
        self.hop_samples = int(hop_duration * sample_rate)

    def load_audio(self, audio_path: str) -> np.ndarray:
        """Nạp và chuẩn hóa âm thanh, không nuốt lỗi âm thầm."""
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {audio_path}")
            
        if audio_path.lower().endswith('.silent'):
            return np.zeros(self.chunk_samples, dtype=np.float32)

        try:
            if audio_path.lower().endswith('.mp3'):
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            else:
                y, sr = sf.read(audio_path, always_2d=False, dtype='float32')
                if len(y.shape) > 1:
                    y = np.mean(y, axis=1)
                if sr != self.sample_rate:
                    y = librosa.resample(y, orig_sr=sr, target_sr=self.sample_rate)
        except Exception as e:
            # Fallback thu doc bang librosa neu soundfile gap loi dinh dang wav
            try:
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            except Exception as final_e:
                print(f"[!] Canh bao: Khong the doc file {audio_path}: {final_e}")
                raise RuntimeError(f"Không thể đọc file âm thanh: {audio_path}") from final_e

        y = np.asarray(y, dtype=np.float32).reshape(-1)
        if y.size == 0:
            raise ValueError(f"File audio rong: {audio_path}")
        if not np.isfinite(y).all():
            raise ValueError(f"Audio chua NaN/Inf: {audio_path}")
        max_val = float(np.max(np.abs(y)))
        if max_val > 1e-6:
            y = y / max_val
        return y

    def slice_into_chunks(self, y: np.ndarray) -> list:
        """Cắt âm thanh thành danh sách các đoạn 10 giây (hop 3s). Trả về ít nhất 1 đoạn."""
        if len(y) <= self.chunk_samples:
            return [np.pad(y, (0, self.chunk_samples - len(y)), mode='constant')]
        chunks = []
        start = 0
        for start in range(0, len(y) - self.chunk_samples + 1, self.hop_samples):
            chunks.append(y[start : start + self.chunk_samples])
        last_chunk_end = start + self.chunk_samples
        remaining_unseen = len(y) - last_chunk_end
        if remaining_unseen > (self.chunk_samples // 3):
            final_chunk = y[-self.chunk_samples:]
            if not np.array_equal(final_chunk, chunks[-1]):
                chunks.append(final_chunk)
        return chunks if chunks else [np.pad(y, (0, max(0, self.chunk_samples - len(y))), mode='constant')[:self.chunk_samples]]


In [ ]:
import scipy.sparse

def shorter(feat: np.ndarray, mean_size: int = 3) -> np.ndarray:
    """Nén trục thời gian của ma trận đặc trưng CQT theo bước mean_size (CoverHunter chuẩn)."""
    if mean_size <= 1:
        return feat
    cqt = feat.T
    height, length = cqt.shape
    new_len = int(length / mean_size)
    new_cqt = np.zeros((height, new_len), dtype=np.float32)
    for i in range(new_len):
        new_cqt[:, i] = cqt[:, i * mean_size : (i + 1) * mean_size].mean(axis=1)
    return new_cqt.T

class PyCqt:
    """Thuật toán CQT Sparse Kernel chính thức của CoverHunter (Tsinghua/MIREX)."""
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = None):
        self._hop_size = hop_size
        self._sample_rate = sample_rate
        if not max_freq:
            max_freq = sample_rate // 2
        self._kernel = self._compute_cqt_kernel(sample_rate, octave_resolution, min_freq, max_freq)

    @staticmethod
    def _compute_cqt_kernel(sampling_frequency: int, octave_resolution: int, minimum_frequency: float, maximum_frequency: float):
        quality_factor = 1.0 / (pow(2.0, 1.0 / octave_resolution) - 1.0)
        number_frequencies = round(octave_resolution * np.log2(maximum_frequency / minimum_frequency))
        fft_length = int(pow(2, np.ceil(np.log2(quality_factor * sampling_frequency / minimum_frequency))))
        cqt_kernel = np.zeros((number_frequencies, fft_length), dtype=complex)

        for i in range(number_frequencies):
            frequency_value = minimum_frequency * pow(2.0, i / octave_resolution)
            window_length = 2 * round(quality_factor * sampling_frequency / frequency_value / 2.0) + 1
            temporal_kernel = (np.hamming(window_length)
                               * np.exp(2.0 * np.pi * 1j * quality_factor
                                        * np.arange(-(window_length - 1) / 2.0, (window_length - 1) / 2.0 + 1) / window_length)
                               / window_length)
            pad_width = int((fft_length - window_length + 1) / 2)
            cqt_kernel[i, pad_width : pad_width + window_length] = temporal_kernel

        cqt_kernel = np.fft.fft(cqt_kernel, axis=1)
        cqt_kernel[np.absolute(cqt_kernel) < 0.01] = 0
        cqt_kernel = scipy.sparse.csr_matrix(cqt_kernel)
        cqt_kernel = np.conjugate(cqt_kernel) / fft_length
        return cqt_kernel

    @staticmethod
    def _compute_cqt_spec(audio_signal: np.ndarray, sampling_frequency: int, time_resolution: int, cqt_kernel):
        step_length = round(sampling_frequency / time_resolution)
        number_times = int(np.floor(len(audio_signal) / step_length))
        number_frequencies, fft_length = np.shape(cqt_kernel)

        pad_left = int(np.ceil((fft_length - step_length) / 2.0))
        pad_right = int(np.floor((fft_length - step_length) / 2.0))
        audio_signal = np.pad(audio_signal, (pad_left, pad_right), "constant", constant_values=(0, 0))

        cqt_spectrogram = np.zeros((number_frequencies, number_times), dtype=np.float32)
        idx = 0
        for j in range(number_times):
            cqt_spectrogram[:, j] = np.absolute(cqt_kernel * np.fft.fft(audio_signal[idx : idx + fft_length]))
            idx += step_length
        return cqt_spectrogram

    def compute_cqt(self, signal_float: np.ndarray, feat_dim_first: bool = False) -> np.ndarray:
        time_resolution = int(round(1.0 / self._hop_size))
        cqt_spectrogram = self._compute_cqt_spec(signal_float, self._sample_rate, time_resolution, self._kernel)
        magnitude = np.maximum(cqt_spectrogram, 1e-9)
        ref_value = max(float(np.max(magnitude)), 1e-4)
        cqt_spectrogram = 20.0 * np.log10(magnitude) - 20.0 * np.log10(ref_value)
        cqt_spectrogram = np.clip(cqt_spectrogram, -80.0, 0.0)
        if not feat_dim_first:
            cqt_spectrogram = cqt_spectrogram.T
        return cqt_spectrogram

class CoverHunterCQT:
    """
    Trích xuất đặc trưng CQT chuẩn CoverHunter:
    - 96 bins (12 bins/octave, 8 octaves từ 32 Hz đến 8000 Hz)
    - hop_size: 0.04s (25 fps), nén thời gian shorter(mean_size=3)
    - Chuẩn hóa dB theo biên độ cực đại: 20 * log10(cqt / max_cqt)
    """
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = 8000.0, mean_size: int = 3):
        self.sample_rate = sample_rate
        self.mean_size = mean_size
        self.py_cqt = PyCqt(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=octave_resolution,
            min_freq=min_freq,
            max_freq=max_freq
        )

    def extract(self, audio: np.ndarray) -> torch.Tensor:
        if isinstance(audio, torch.Tensor):
            audio = audio.detach().cpu().numpy()
        feat = self.py_cqt.compute_cqt(audio, feat_dim_first=False) # (T, 96)
        if self.mean_size > 1:
            feat = shorter(feat, mean_size=self.mean_size)
        return torch.from_numpy(feat).float()
class CQTExtractor:
    """Trích xuất Constant-Q Transform (CQT) hỗ trợ cả chế độ Fixed-Length (83 frames) và Variable-Length."""
    def __init__(self, sample_rate: int = 16000, n_bins: int = 96, bins_per_octave: int = 12, hop_size: float = 0.04, mean_size: int = 3, target_frames: int = 83):
        self.sample_rate = sample_rate
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave
        self.hop_size = hop_size
        self.mean_size = mean_size
        self.target_frames = target_frames
        self.cqt_impl = CoverHunterCQT(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=bins_per_octave,
            min_freq=32.0,
            max_freq=8000.0,
            mean_size=mean_size
        )

    def extract(self, y: np.ndarray, variable_length: bool = False) -> torch.Tensor:
        """
        Trích xuất tensor CQT từ tín hiệu âm thanh 1D.
        - variable_length=False: Căn chỉnh về đúng [1, 96, target_frames] (mặc định 83 frames ~ 10s).
        - variable_length=True: Giữ nguyên độ dài thời gian thực tế [1, 96, T] phục vụ Stage 2 DTW.
        """
        feat = self.cqt_impl.extract(y)  # (T, 96)
        tensor = feat.T.unsqueeze(0)      # (1, 96, T)

        if not variable_length and self.target_frames is not None and self.target_frames > 0:
            curr_t = tensor.shape[-1]
            min_val = tensor.min()
            if curr_t < self.target_frames:
                tensor = F.pad(tensor, (0, self.target_frames - curr_t), mode='constant', value=min_val.item())
            elif curr_t > self.target_frames:
                tensor = tensor[:, :, :self.target_frames]
        return tensor



In [ ]:
def apply_cqt_pitch_shift(cqt: torch.Tensor, shift_bins: int) -> torch.Tensor:
    """
    Dịch chuyển cao độ trên ma trận CQT bằng cách tịnh tiến trục tần số (dim -2).
    - shift_bins > 0: Dịch lên cao độ cao hơn (tăng semitone).
    - shift_bins < 0: Dịch xuống cao độ thấp hơn (giảm semitone).
    - Điền các bin trống bằng cqt.min() (mức sàn im lặng), KHÔNG điền 0.0 (vốn là 0 dB cực đại).
    """
    if shift_bins == 0:
        return cqt

    min_val = cqt.min()
    shifted = torch.full_like(cqt, fill_value=min_val)
    num_bins = cqt.shape[-2]

    if shift_bins > 0:
        if shift_bins < num_bins:
            shifted[..., shift_bins:, :] = cqt[..., :num_bins - shift_bins, :]
    else:
        abs_shift = abs(shift_bins)
        if abs_shift < num_bins:
            shifted[..., :num_bins - abs_shift, :] = cqt[..., abs_shift:, :]

    return shifted
class CQTPitchShiftAugmentation:
    """
    Tăng cường dữ liệu dịch tone ngẫu nhiên (Full-Octave Pitch-Shift Data Augmentation).
    Dịch ngẫu nhiên trong dải [-max_shift_bins, +max_shift_bins] nửa cung (semitones).
    Mặc định dải ±12 semitones bao trọn 1 quãng 8 (chuyển đổi hoàn hảo giữa giọng Nam và Nữ).
    """
    def __init__(self, max_shift_bins: int = 12, prob: float = 0.8):
        self.max_shift_bins = max_shift_bins
        self.prob = prob

    def __call__(self, cqt_tensor: torch.Tensor) -> torch.Tensor:
        if random.random() > self.prob:
            return cqt_tensor
        shift = random.randint(-self.max_shift_bins, self.max_shift_bins)
        return apply_cqt_pitch_shift(cqt_tensor, shift)
class SpectrogramTempoAndMaskDistortion:
    """Nonlinear time warp plus SpecAugment masks; the CQT frequency axis is never warped."""
    def __init__(self, time_scale_range=(0.5, 1.5), max_mask_t=0, max_mask_f=0, prob=0.6, knots=6):
        self.scale_range = time_scale_range
        self.max_mask_t = int(max_mask_t)
        self.max_mask_f = int(max_mask_f)
        self.prob = prob
        self.knots = knots
        if not 0 < time_scale_range[0] <= time_scale_range[1] or knots < 2:
            raise ValueError('Invalid distortion parameters')
        if self.max_mask_t < 0 or self.max_mask_f < 0:
            raise ValueError('Mask widths must be non-negative')

    def _axis(self, size, batch, device, dtype):
        increments = torch.empty(batch, 1, self.knots, device=device, dtype=dtype).uniform_(*self.scale_range)
        positions = torch.cat([torch.zeros(batch, 1, 1, device=device, dtype=dtype), increments.cumsum(-1)], -1)
        positions = 2 * positions / positions[..., -1:] - 1
        return F.interpolate(positions, size=size, mode='linear', align_corners=True)[:, 0]

    @staticmethod
    def _mask_axis(x, axis, max_width):
        if max_width <= 0 or x.shape[axis] <= 1:
            return
        width = random.randint(1, min(max_width, x.shape[axis] - 1))
        start = random.randint(0, x.shape[axis] - width)
        slices = [slice(None)] * x.ndim
        slices[axis] = slice(start, start + width)
        fill_value = x.amin(dim=(-2, -1), keepdim=True)
        x[tuple(slices)] = fill_value

    def __call__(self, spec):
        if random.random() >= self.prob:
            return spec
        was_unbatched = spec.ndim == 3
        x = spec.unsqueeze(0) if was_unbatched else spec
        b, _, h, w = x.shape
        tx = self._axis(w, b, x.device, x.dtype)
        fy = torch.linspace(-1, 1, h, device=x.device, dtype=x.dtype).expand(b, h)
        grid = torch.stack([tx[:, None, :].expand(b, h, w), fy[:, :, None].expand(b, h, w)], -1)
        out = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True).clone()
        self._mask_axis(out, axis=-1, max_width=self.max_mask_t)
        self._mask_axis(out, axis=-2, max_width=self.max_mask_f)
        return out.squeeze(0) if was_unbatched else out


In [ ]:
from torch.autograd import Function

class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_val: float):
        ctx.lambda_val = lambda_val
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    """Lớp đảo ngược gradient với hệ số lambda điều chỉnh động theo epoch."""
    def __init__(self, lambda_val: float = 0.2):
        super(GradientReversalLayer, self).__init__()
        self.lambda_val = lambda_val

    def set_lambda(self, lambda_val: float):
        self.lambda_val = lambda_val

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_val)
class LocalDomainClassifier(nn.Module):
    """
    Bộ phân loại miền cục bộ (Local Domain Classifier) theo kiến trúc ByteHum:
    - Nhận đầu vào: Chuỗi đặc trưng cục bộ x [B, T, 128] từ Conformer encoder.
    - Áp dụng các tầng tích chập 1D theo trục thời gian để học đặc trưng miền cục bộ.
    - Dự đoán phân phối nhãn miền (0: Vocal phong thu, 1: Tiếng humming) trên từng phân đoạn thời gian.
    """
    def __init__(self, feature_dim: int = 128, hidden_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, 128] -> transpose thành [B, 128, T] cho Conv1D
        x_t = x.transpose(1, 2)
        feat = self.conv(x_t) # [B, hidden_dim, T]
        # Frame-level logits: [B, T]
        frame_logits = self.fc(feat.transpose(1, 2)).squeeze(-1)
        # Tổng hợp logits qua trục thời gian T
        return frame_logits.mean(dim=-1) # [B]
def build_coverhunter_model(ckpt_path: str = None, device: str = "cpu", allow_random_init: bool = False):
    """
    Khởi tạo mô hình CoverHunter Conformer chính thức (9.91M tham số, 6 khối Conformer, Attention Pooling 128-D).
    Nạp trọn vẹn 265/265 tensor từ checkpoint SHS100K pre-trained.
    """
    # Cấu hình siêu tham số mặc định chuẩn xác của CoverHunter SHS100K
    hp = {
        'input_dim': 96, 'embed_dim': 128,
        'encoder': {'output_dims': 128, 'num_blocks': 6, 'attention_dim': 256},
        'pool_type': 'attention',
        'ce': {'output_dims': 30000, 'weight': 1.0, 'gamma': 2},
        'triplet': {'margin': 0.3, 'weight': 0.1},
        'center': {'weight': 0.0}
    }

    ch_model = Model(hp)

    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state_dict = ckpt.get("generator", ckpt.get("model_state_dict", ckpt))
        ch_model.load_state_dict(state_dict)
        total_p = sum(p.numel() for p in ch_model.parameters())
        print(f"[+] Nạp thành công 265/265 tensors ({total_p:,} params) từ CoverHunter Pretrained:\n    -> {ckpt_path}")
    elif allow_random_init:
        print("[*] Khoi tao backbone tam thoi; checkpoint fine-tuned se duoc nap ngay sau do.")
    else:
        raise FileNotFoundError(f"Bat buoc phai co checkpoint Stage 1 SHS100K: {ckpt_path}")

    ch_model = ch_model.to(device)
    return ch_model
class LocalAlignmentClassifier(nn.Module):
    """
    Local Alignment Classifier (LAC) theo kiến trúc ByteHum (ICASSP 2024):
    - Trọng số Prototype: W in R^{K x L x C}, trong đó K là số lớp bài hát (train songs),
      L là số phân đoạn prototype cục bộ (mặc định L=4), C=128.
    - Điểm tương đồng MaxMean: S(X, W_k) = (1/L) * sum_{l=1}^L max_{t=1}^T cos(x_t, w_{k,l}).
    - Cho phép căn chỉnh cục bộ bất biến thời gian và trôi dạt nhịp điệu giữa Humming và Vocal.
    """
    def __init__(self, num_classes: int = 210, feature_dim: int = 128, num_prototypes: int = 4):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_prototypes = num_prototypes
        # Khởi tạo ma trận prototype W [K, L, C]
        self.prototypes = nn.Parameter(torch.randn(num_classes, num_prototypes, feature_dim) * 0.05)
        self.temperature = nn.Parameter(torch.tensor(10.0))

    def forward(self, local_feat: torch.Tensor) -> torch.Tensor:
        # local_feat: [B, T, 128]
        # Chuẩn hóa L2 trên trục đặc trưng 128-D
        x_norm = F.normalize(local_feat, p=2, dim=-1) # [B, T, C]
        w_norm = F.normalize(self.prototypes, p=2, dim=-1) # [K, L, C]

        # Tính cosine similarity giữa toàn bộ T frames và K*L prototypes qua phép nhân ma trận:
        w_flat = w_norm.view(-1, self.feature_dim).t() # [C, K*L]
        sim = torch.matmul(x_norm, w_flat) # [B, T, K*L]
        B, T, _ = sim.shape
        sim = sim.view(B, T, self.num_classes, self.num_prototypes) # [B, T, K, L]

        # Phep toan MaxMean: Max theo truc thoi gian T, Mean theo so prototypes L
        max_t, _ = sim.max(dim=1) # [B, K, L]
        logits = max_t.mean(dim=-1) # [B, K]
        return logits * self.temperature.clamp(min=1.0, max=100.0)

class ConformerByteHumModel(nn.Module):
    """
    MÔ HÌNH HOÀN CHỈNH CONFORMER-BYTEHUM CHO QBH DOMAIN ADAPTATION:
    - Backbone: Conformer (6 khối, Conv2d Subsampling 4x -> T_enc=20 frames cho 10s audio)
    - Output 1: global_emb [B, 128] (qua Attentive Statistics Pooling + L2 Normalize)
    - Output 2: local_feat [B, T=20, 128] (chuỗi đặc trưng cục bộ trước pooling)
    - Output 3: lac_logits [B, K] (Local Alignment Classifier phân loại bài hát qua MaxMean)
    - Output 4: domain_logits [B] (Local Domain Classifier phân biệt miền Vocal vs Humming)
    """
    def __init__(self, ckpt_path: str = None, input_dim: int = 96, embedding_dim: int = 128, num_classes: int = 210, lambda_grl: float = 0.2, allow_random_init: bool = False):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        if embedding_dim not in (128, 256, 512):
            raise ValueError('embedding_dim must be 128, 256 or 512')
        self.local_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)
        self.global_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)

        # Nạp backbone Conformer pre-trained từ CoverHunter Stage 1
        self.feature_extractor = build_coverhunter_model(ckpt_path=ckpt_path, allow_random_init=allow_random_init)
        
        # Đóng băng các tầng phụ không dùng của Stage 1 (ce_layer, center_loss)
        if hasattr(self.feature_extractor, '_ce_layer'):
            for p in self.feature_extractor._ce_layer.parameters(): p.requires_grad = False
        if hasattr(self.feature_extractor, '_center_loss'):
            for p in self.feature_extractor._center_loss.parameters(): p.requires_grad = False

        self.grl = GradientReversalLayer(lambda_val=lambda_grl)
        self.domain_classifier = LocalDomainClassifier(feature_dim=embedding_dim, hidden_dim=64)
        self.lac = LocalAlignmentClassifier(num_classes=num_classes, feature_dim=embedding_dim, num_prototypes=4)

    def train(self, mode: bool = True):
        super().train(mode)
        # Preserve the SHS100K CMVN statistics while fine-tuning on humming audio.
        if mode and hasattr(self.feature_extractor, '_global_cmvn'):
            self.feature_extractor._global_cmvn.eval()
        return self

    def set_lambda(self, lambda_val: float):
        self.grl.set_lambda(lambda_val)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4: x = x.squeeze(1)
        if x.dim() == 3:
            if x.shape[1] == 96 and x.shape[2] != 96: x = x.transpose(1, 2)
        elif x.dim() == 2:
            if x.shape[0] == 96 and x.shape[1] != 96: x = x.t().unsqueeze(0)
            else: x = x.unsqueeze(0)
        return x

    def extract_backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trích xuất chuỗi đặc trưng cục bộ [B, T=20, 128] từ Conformer encoder trước ASP pooling."""
        x_prep = self._prepare_input(x)
        x_cmvn = self.feature_extractor._global_cmvn(x_prep.transpose(1, 2)).transpose(1, 2)
        xs_lens = torch.full([x_cmvn.size(0)], fill_value=x_cmvn.size(1), dtype=torch.long, device=x_cmvn.device)
        x_enc, _ = self.feature_extractor._encoder(x_cmvn, xs_lens=xs_lens, decoding_chunk_size=-1)
        return x_enc # [B, T=20, 128]

    def extract_local_features(self, x):
        return self.local_projection(self.extract_backbone_features(x))

    def extract_global_embedding(self, local_feat: torch.Tensor) -> torch.Tensor:
        """Chiếu chuỗi đặc trưng cục bộ qua Attentive Statistics Pooling thành Global 128-D Embedding."""
        emb = self.feature_extractor._pool_layer(local_feat)
        return F.normalize(self.global_projection(emb), p=2, dim=-1)

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Hàm tiện ích trích xuất trực tiếp Global Embedding 128-D từ phổ CQT đầu vào."""
        return self.extract_global_embedding(self.extract_backbone_features(x))

    def forward(self, x: torch.Tensor, compute_lac: bool = True, compute_domain: bool = True):
        """Return global/local features and only the auxiliary heads requested by the run."""
        backbone_feat = self.extract_backbone_features(x)
        local_feat = self.local_projection(backbone_feat)
        global_emb = self.extract_global_embedding(backbone_feat)
        lac_logits = self.lac(local_feat) if compute_lac else None
        domain_logits = self.domain_classifier(self.grl(local_feat)) if compute_domain else None
        return global_emb, local_feat, lac_logits, domain_logits

    def freeze_lower_layers(self, num_blocks_to_freeze: int = 3, freeze_frontend: bool = True):
        """Đóng băng tầng frontend và các khối Conformer đầu tiên theo chiến lược Deep Freeze."""
        if freeze_frontend:
            if hasattr(self.feature_extractor, '_global_cmvn'):
                for p in self.feature_extractor._global_cmvn.parameters(): p.requires_grad = False
            if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'embed'):
                for p in self.feature_extractor._encoder.embed.parameters(): p.requires_grad = False

        if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'encoders'):
            for i, block in enumerate(self.feature_extractor._encoder.encoders):
                if i < num_blocks_to_freeze:
                    for p in block.parameters(): p.requires_grad = False
                else:
                    for p in block.parameters(): p.requires_grad = True

        if hasattr(self.feature_extractor, '_pool_layer'):
            for p in self.feature_extractor._pool_layer.parameters(): p.requires_grad = True
        for p in self.domain_classifier.parameters(): p.requires_grad = True
        for p in self.lac.parameters(): p.requires_grad = True

In [ ]:
class CrossBatchMemoryBank:
    """FIFO memory bank tren CPU; embeddings trong bank khong mang gradient."""
    def __init__(self, capacity=4096):
        self.capacity = int(capacity)
        self.embeddings = None
        self.song_ids = None

    def __len__(self):
        return 0 if self.song_ids is None else int(self.song_ids.numel())

    def clear(self):
        self.embeddings = None
        self.song_ids = None

    @torch.no_grad()
    def enqueue(self, embeddings, song_ids):
        new_embeddings = F.normalize(embeddings.detach(), p=2, dim=-1).cpu()
        new_song_ids = song_ids.detach().long().cpu()
        if self.embeddings is None:
            self.embeddings = new_embeddings[-self.capacity:]
            self.song_ids = new_song_ids[-self.capacity:]
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)[-self.capacity:]
            self.song_ids = torch.cat([self.song_ids, new_song_ids], dim=0)[-self.capacity:]

    def candidates(self, device, dtype):
        if self.embeddings is None:
            return None, None
        return self.embeddings.to(device=device, dtype=dtype), self.song_ids.to(device=device)


def mining_hardest_ratio(epoch):
    """Curriculum: semi-hard first, then gradually emphasize hardest negatives."""
    if epoch <= 5:
        return 0.0
    if epoch <= 15:
        return 0.3
    return 0.7


def _topk_masked_mean(values, mask, top_k):
    masked = values.masked_fill(~mask, float('-inf'))
    k = min(int(top_k), masked.shape[1])
    top_values = torch.topk(masked, k=k, dim=1).values
    finite = torch.isfinite(top_values)
    total = torch.where(finite, top_values, torch.zeros_like(top_values)).sum(dim=1, keepdim=True)
    count = finite.sum(dim=1, keepdim=True)
    return total / count.clamp_min(1), count > 0


def in_batch_cosine_triplet_loss(
    emb_a,
    emb_p,
    emb_n,
    song_ids,
    neg_song_ids,
    margin=0.20,
    memory_bank=None,
    hardest_ratio=0.0,
    top_k=5
):
    """Semi-hard/hard curriculum over in-batch candidates plus a cross-batch memory bank."""
    cos_pos = torch.sum(emb_a * emb_p, dim=-1, keepdim=True)
    candidate_embeddings = [emb_n, emb_p, emb_a]
    candidate_song_ids = [neg_song_ids, song_ids, song_ids]

    if memory_bank is not None:
        memory_embeddings, memory_song_ids = memory_bank.candidates(emb_a.device, emb_a.dtype)
        if memory_embeddings is not None:
            candidate_embeddings.append(memory_embeddings)
            candidate_song_ids.append(memory_song_ids)

    candidates = torch.cat(candidate_embeddings, dim=0)
    candidate_labels = torch.cat(candidate_song_ids, dim=0)
    similarities = torch.mm(emb_a, candidates.T)
    valid_negative = song_ids.unsqueeze(1) != candidate_labels.unsqueeze(0)

    hard_similarity, has_hard = _topk_masked_mean(similarities, valid_negative, top_k)
    semi_hard_mask = (
        valid_negative
        & (similarities < cos_pos)
        & (similarities > (cos_pos - margin))
    )
    semi_similarity, has_semi = _topk_masked_mean(similarities, semi_hard_mask, top_k)
    semi_or_fallback = torch.where(has_semi, semi_similarity, hard_similarity)
    selected_negative = (1.0 - hardest_ratio) * semi_or_fallback + hardest_ratio * hard_similarity

    if not bool(has_hard.all()):
        raise RuntimeError("Batch khong co negative hop le sau khi mask song ID")
    return F.relu(selected_negative - cos_pos + margin).mean()

# Tiêu chuẩn mất mát cho Domain Adaptation và Local Alignment
criterion_domain = nn.BCEWithLogitsLoss()
criterion_lac = nn.CrossEntropyLoss()

print("[+] Loss weights are controlled only by RUN_CONFIG; domain BCE uses weight 1.0 and GRL controls the backbone gradient.")
def prototype_diversity_loss(prototypes):
    """Discourage LAC prototypes of the same song from collapsing to one direction."""
    normalized = F.normalize(prototypes, p=2, dim=-1)
    gram = torch.matmul(normalized, normalized.transpose(-1, -2))
    eye = torch.eye(gram.shape[-1], device=gram.device, dtype=gram.dtype).unsqueeze(0)
    off_diagonal = (gram - eye) * (1.0 - eye)
    return off_diagonal.square().sum() / max(1, gram.shape[0] * gram.shape[1] * (gram.shape[1] - 1))


def local_alignment_triplet(
    anchor, positive, negative, song_ids, negative_song_ids,
    margin=0.2, hardest_ratio=0.0, top_k=5
):
    """Fully vectorized query-directed MaxMean with semi-hard-to-hard curriculum."""
    anchor, positive, negative = [F.normalize(x, dim=-1) for x in (anchor, positive, negative)]
    positive_scores = torch.bmm(anchor, positive.transpose(1, 2)).max(-1).values.mean(-1, keepdim=True)
    candidates = torch.cat([positive, negative], dim=0)
    labels = torch.cat([song_ids, negative_song_ids], dim=0)
    valid = song_ids.unsqueeze(1) != labels.unsqueeze(0)
    if not bool(valid.any(dim=1).all()):
        raise ValueError('LAT requires different-song negatives for every anchor')

    similarities = torch.einsum('btc,nsc->bnts', anchor, candidates).max(dim=-1).values.mean(dim=-1)
    hard_score, has_hard = _topk_masked_mean(similarities, valid, top_k)
    semi_mask = valid & (similarities < positive_scores) & (similarities > positive_scores - margin)
    semi_score, has_semi = _topk_masked_mean(similarities, semi_mask, top_k)
    semi_or_fallback = torch.where(has_semi, semi_score, hard_score)
    selected = (1.0 - hardest_ratio) * semi_or_fallback + hardest_ratio * hard_score
    if not bool(has_hard.all()):
        raise RuntimeError('LAT has no valid negative')
    return F.relu(selected - positive_scores + margin).mean()


In [ ]:
class SongBalancedBatchSampler:
    """Tao moi batch theo cau truc P bai hat x K query de tang negative hop le."""
    def __init__(self, dataset, songs_per_batch=8, samples_per_song=2):
        self.dataset = dataset
        self.songs_per_batch = songs_per_batch
        self.samples_per_song = samples_per_song
        self.batch_size = songs_per_batch * samples_per_song
        self.song_to_indices = {}
        for index, (song, _, _) in enumerate(dataset.humming_samples):
            self.song_to_indices.setdefault(song, []).append(index)
        self.songs = sorted(self.song_to_indices)
        if len(self.songs) < songs_per_batch:
            raise ValueError(f"Can it nhat {songs_per_batch} bai train, chi tim thay {len(self.songs)}")
        self.num_batches = max(1, len(dataset) // self.batch_size)

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        song_order = self.songs.copy()
        random.shuffle(song_order)
        cursor = 0
        for _ in range(self.num_batches):
            if cursor + self.songs_per_batch > len(song_order):
                random.shuffle(song_order)
                cursor = 0
            selected_songs = song_order[cursor:cursor + self.songs_per_batch]
            cursor += self.songs_per_batch
            batch = []
            for song in selected_songs:
                indices = self.song_to_indices[song]
                if len(indices) >= self.samples_per_song:
                    batch.extend(random.sample(indices, self.samples_per_song))
                else:
                    batch.extend(random.choices(indices, k=self.samples_per_song))
            random.shuffle(batch)
            yield batch



In [ ]:
@torch.inference_mode()
def score_gallery(query_global, query_local, gallery):
    # Bound padding to one song group, and similarities to reference-frame blocks.
    qg, ql = query_global.to(device), query_local.to(device)
    songs = list(gallery)
    local_scores, global_scores = [], []
    for offset in range(0, len(songs), GALLERY_SONG_BLOCK):
        group = [gallery[s] for s in songs[offset:offset + GALLERY_SONG_BLOCK]]
        gp = nn.utils.rnn.pad_sequence([g for g, _ in group], batch_first=True).to(device)
        gm = torch.arange(gp.shape[1], device=device)[None, :] < torch.tensor([len(g) for g, _ in group], device=device)[:, None]
        similarities = torch.einsum('qd,sgd->sqg', qg, gp).masked_fill(~gm[:, None, :], float('-inf'))
        global_scores.append(similarities.amax(dim=(1, 2)).cpu())
        max_per_query = torch.full((len(group), len(ql)), float('-inf'), device=device)
        longest = max(len(l) for _, l in group)
        for frame_start in range(0, longest, REFERENCE_FRAME_BLOCK):
            slices = [l[frame_start:frame_start + REFERENCE_FRAME_BLOCK] for _, l in group]
            padded = nn.utils.rnn.pad_sequence(slices, batch_first=True).to(device)
            mask = torch.arange(padded.shape[1], device=device)[None, :] < torch.tensor([len(l) for l in slices], device=device)[:, None]
            sims = torch.einsum('qd,sgd->sqg', ql, padded).masked_fill(~mask[:, None, :], float('-inf'))
            max_per_query = torch.maximum(max_per_query, sims.amax(dim=-1))
        local_scores.append(max_per_query.mean(dim=-1).cpu())
    local, global_ = torch.cat(local_scores), torch.cat(global_scores)
    return {'local': local, 'global': global_, 'fused': 0.5 * (local + global_)}



In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,
                                     separators=(',', ':')).encode()).hexdigest()

FILE_HASHES = {}
def file_sha256(path):
    path = Path(path).resolve()
    stat = path.stat()
    key = (str(path), stat.st_size, stat.st_mtime_ns)
    if key not in FILE_HASHES:
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for part in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(part)
        FILE_HASHES[key] = h.hexdigest()
    return FILE_HASHES[key]

def atomic_json(path, data):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temp, path)

def atomic_torch(path, data):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    torch.save(data, temp)
    os.replace(temp, path)

def chunk_starts(length, window, hop):
    if min(length, window, hop) < 1 or hop > window:
        raise ValueError('Invalid audio length/window/hop')
    if length <= window:
        return [0]
    starts = list(range(0, length - window + 1, hop))
    if starts[-1] + window < length:
        starts.append(length - window)
    return starts

class FullAudioPreprocessor(AudioPreprocessor):
    def slice_into_chunks(self, y):
        starts = chunk_starts(len(y), self.chunk_samples, self.hop_samples)
        return [np.pad(y[start:start + self.chunk_samples],
                       (0, max(0, self.chunk_samples - len(y[start:start + self.chunk_samples]))))
                for start in starts]

prep = FullAudioPreprocessor()
cqt_ext = CQTExtractor()

class BoundedCache(OrderedDict):
    def __init__(self, limit_mb=512):
        super().__init__()
        self.limit = limit_mb * 1024 ** 2
        self.bytes = 0

    @staticmethod
    def size(value):
        return sum(t.numel() * t.element_size() for t in value)

    def __getitem__(self, key):
        value = super().__getitem__(key)
        self.move_to_end(key)
        return value

    def __setitem__(self, key, value):
        if key in self:
            self.bytes -= self.size(super().pop(key))
        size = self.size(value)
        if size > self.limit:
            return
        while self and self.bytes + size > self.limit:
            self.bytes -= self.size(self.popitem(last=False)[1])
        super().__setitem__(key, value)
        self.bytes += size

CQT_RAM = BoundedCache()
CQT_DIR = OUTPUT_ROOT / 'cache' / 'cqt'
FEATURE_DIR = OUTPUT_ROOT / 'cache' / 'features'
CQT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

def raw_cqts(path):
    key = digest(dict(audio=file_sha256(path), prep=PREPROCESS_CONFIG,
                      code=SOURCE_CODE_DIGEST, versions=RUNTIME_VERSIONS))
    if key in CQT_RAM:
        return CQT_RAM[key]
    cache_path = CQT_DIR / (key + '.pt')
    if cache_path.is_file():
        chunks = torch.load(cache_path, map_location='cpu', weights_only=True)
    else:
        y = prep.load_audio(str(path))
        if np.max(np.abs(y)) <= 1e-6:
            raise ValueError(f'Silent audio is not a valid query/reference: {path}')
        chunks = [cqt_ext.extract(x) for x in prep.slice_into_chunks(y)]
        atomic_torch(cache_path, chunks)
    if not chunks or any(x.shape != (1, 96, 83) or not torch.isfinite(x).all() for x in chunks):
        raise ValueError(f'Invalid CQT cache: {cache_path}')
    CQT_RAM[key] = chunks
    return chunks

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
SOURCE_SONG_COUNTS = {'train': 210, 'val': 45, 'test': 45}
SOURCE_QUERY_COUNTS = {'train': 3612, 'val': 766}
AVAILABLE_QUERY_COUNTS = {'train': 3583, 'val': 763, 'test': 802}
SOURCE_MAPPING_SHA256 = 'e0d2094858e44cbec76a9c87fb8d8909cdfb7350f905494a0cb6cda22c9ffb6c'
SOURCE_SPLIT_MANIFEST_SHA256 = '06fcd81f0befa48cde9fbce2bbae92c9579ac295bd9747ce85fb8fe28b65229d'
LOCKED_MANIFEST_SHA256 = '38fa98e0157e8bff366f4b6c69e7bf0b193148882b165b00b6fae6bec74efe63'
SOURCE_QUERY_ROWS = json.loads(zlib.decompress(base64.b64decode(SOURCE_QUERY_INDEX_ZLIB_B64)).decode('utf-8'))
_source_index_bytes = json.dumps(SOURCE_QUERY_ROWS, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
if hashlib.sha256(_source_index_bytes).hexdigest() != SOURCE_QUERY_INDEX_SHA256:
    raise ValueError('Embedded verified source-query index failed its SHA256 check.')
SOURCE_QUERY_BY_ID = {row['audio_id']: row for row in SOURCE_QUERY_ROWS}
if len(SOURCE_QUERY_ROWS) != 5187 or len(SOURCE_QUERY_BY_ID) != 5187:
    raise ValueError('Embedded source-query index must contain 5,187 unique query IDs.')

def repair_legacy_dataset_split():
    """Copy read-only input to working storage, then route songs by verified query IDs."""
    global DATASET_SPLIT_DIR
    source_split = Path(DATASET_SPLIT_DIR).resolve()
    output_root = Path(OUTPUT_ROOT).resolve()
    try:
        source_split.relative_to(output_root)
        source_is_writable = True
    except ValueError:
        source_is_writable = False

    if not source_is_writable:
        working_root = output_root / 'dataset_working_copy'
        working_split = working_root / 'datasets' / 'dataset_split'
        marker = working_root / '.qbh_source_copy.json'
        if working_root.exists():
            if not marker.is_file():
                raise RuntimeError(
                    f'Incomplete dataset working copy: {working_root}. Choose a fresh OUTPUT_ROOT and rerun.'
                )
            marker_data = json.loads(marker.read_text(encoding='utf-8'))
            if marker_data.get('source_split') != str(source_split) or not working_split.is_dir():
                raise RuntimeError(
                    f'Dataset working copy does not match this Kaggle input: {working_root}. '
                    'Choose a fresh OUTPUT_ROOT and rerun.'
                )
        else:
            required_bytes = sum(p.stat().st_size for p in source_split.rglob('*') if p.is_file())
            free_bytes = shutil.disk_usage(output_root).free
            reserve_bytes = 1024 ** 3
            if required_bytes + reserve_bytes > free_bytes:
                raise OSError(
                    f'Not enough writable space to repair the read-only Kaggle input. '
                    f'Need about {required_bytes / 1024**3:.2f} GiB plus 1 GiB free; '
                    f'available {free_bytes / 1024**3:.2f} GiB at {output_root}.'
                )
            temp_root = output_root / '.dataset_working_copy.tmp'
            if temp_root.exists():
                if not temp_root.resolve().is_relative_to(output_root):
                    raise RuntimeError(f'Unsafe temporary dataset path: {temp_root}')
                shutil.rmtree(temp_root)
            (temp_root / 'datasets').mkdir(parents=True, exist_ok=True)
            shutil.copytree(source_split, temp_root / 'datasets' / 'dataset_split')
            (temp_root / '.qbh_source_copy.json').write_text(
                json.dumps({'source_split': str(source_split)}, ensure_ascii=False, indent=2),
                encoding='utf-8'
            )
            os.replace(temp_root, working_root)
        DATASET_SPLIT_DIR = working_split
        print(f'Writable dataset copy: {DATASET_SPLIT_DIR}', flush=True)

    expected_manifest_bytes = zlib.decompress(base64.b64decode(VERIFIED_SPLIT_MANIFEST_ZLIB_B64))
    expected_locked_bytes = zlib.decompress(base64.b64decode(VERIFIED_LOCKED_HOLDOUT_ZLIB_B64))
    if hashlib.sha256(expected_manifest_bytes).hexdigest() != SOURCE_SPLIT_MANIFEST_SHA256:
        raise ValueError('Embedded verified split manifest failed its SHA256 check.')
    if hashlib.sha256(expected_locked_bytes).hexdigest() != LOCKED_MANIFEST_SHA256:
        raise ValueError('Embedded locked holdout manifest failed its SHA256 check.')
    expected_manifest = json.loads(expected_manifest_bytes.decode('utf-8'))
    by_song_id = {row['song_id']: (folder, row) for folder, row in expected_manifest.items()}
    if len(by_song_id) != 300 or len(expected_manifest) != 300:
        raise ValueError('Embedded source split must map 300 unique songs.')

    folder_to_song = {}
    for split in ('train', 'val', 'test'):
        split_dir = DATASET_SPLIT_DIR / split
        if not split_dir.is_dir():
            raise FileNotFoundError(f'Missing legacy split directory: {split_dir}')
        for folder in (p for p in split_dir.iterdir() if p.is_dir()):
            for path in folder.rglob('*.wav'):
                query = SOURCE_QUERY_BY_ID.get(path.stem)
                if query is None:
                    continue
                sid = query['song_id']
                previous = folder_to_song.setdefault(folder.name, sid)
                if previous != sid:
                    raise ValueError(f'Song folder contains query IDs from multiple songs: {folder}')
    if set(folder_to_song) != set(expected_manifest):
        missing = sorted(set(expected_manifest) - set(folder_to_song))
        extra = sorted(set(folder_to_song) - set(expected_manifest))
        raise ValueError(f'Cannot map all legacy song folders by query ID; missing={missing[:5]}, extra={extra[:5]}')

    moves = []
    for folder, sid in folder_to_song.items():
        expected_folder, metadata = by_song_id.get(sid, (None, None))
        if expected_folder != folder or metadata is None:
            raise ValueError(f'Legacy folder-to-song mapping differs from verified provenance: {folder}')
        found = [DATASET_SPLIT_DIR / split / folder for split in ('train', 'val', 'test')
                 if (DATASET_SPLIT_DIR / split / folder).is_dir()]
        if len(found) != 1:
            raise ValueError(f'Expected one directory for song {folder}, found {len(found)}')
        source = found[0]
        target = DATASET_SPLIT_DIR / metadata['split'] / folder
        if source != target:
            if target.exists():
                raise FileExistsError(f'Split repair destination already exists: {target}')
            moves.append((source, target))
    for source, target in moves:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source), str(target))

    # These files are written only inside the extracted Kaggle working copy.
    manifest_path = DATASET_SPLIT_DIR / 'split_manifest.json'
    manifest_tmp = manifest_path.with_suffix('.json.tmp')
    manifest_tmp.write_bytes(expected_manifest_bytes)
    os.replace(manifest_tmp, manifest_path)
    locked_path = DATASET_SPLIT_DIR.parent / 'holdout' / 'locked_final_test_manifest.csv'
    locked_path.parent.mkdir(parents=True, exist_ok=True)
    locked_tmp = locked_path.with_suffix('.csv.tmp')
    locked_tmp.write_bytes(expected_locked_bytes)
    os.replace(locked_tmp, locked_path)
    qbh_success(f'Repaired legacy song folders by verified query IDs; routed {len(moves)} folders.')

def validate_dataset_package(split_ids):
    split_manifest_path = DATASET_SPLIT_DIR / 'split_manifest.json'
    if not split_manifest_path.is_file():
        raise FileNotFoundError('Missing dataset_split/split_manifest.json; use the rebuilt source-split dataset.')
    manifest_bytes = split_manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != SOURCE_SPLIT_MANIFEST_SHA256:
        raise ValueError('dataset_split/split_manifest.json is not the verified source-split manifest.')
    split_manifest = json.loads(manifest_bytes.decode('utf-8'))
    if len(split_manifest) != 300:
        raise ValueError(f'Source split manifest must map all 300 songs; found {len(split_manifest)}.')
    song_id_values = [row.get('song_id') for row in split_manifest.values()]
    if any(not isinstance(sid, str) or len(sid) != 16 for sid in song_id_values) or len(set(song_id_values)) != 300:
        raise ValueError('Source split manifest has missing or duplicate song IDs; rebuild it from the verified mapping.')
    mapping_counts = Counter(row.get('split') for row in split_manifest.values())
    if mapping_counts != Counter(SOURCE_SONG_COUNTS):
        raise ValueError(f'Wrong verified source-song split counts: {dict(mapping_counts)}')
    for split, songs in split_ids.items():
        if len(songs) != SOURCE_SONG_COUNTS[split]:
            raise ValueError(f'Wrong source split size for {split}: expected {SOURCE_SONG_COUNTS[split]}, found {len(songs)}.')
        for song in songs:
            row = split_manifest.get(song)
            if not row or row.get('song_id') not in set(song_id_values) or row.get('split') != split:
                raise ValueError(f'Song folder does not match source split manifest: split={split}, folder={song}.')

    source_song_split = {}
    source_query_totals = Counter()
    source_song_query_totals = Counter()
    for row in SOURCE_QUERY_ROWS:
        sid, source_split = row['song_id'], row['split']
        previous = source_song_split.setdefault(sid, source_split)
        if previous != source_split:
            raise ValueError(f'Embedded source-query index has conflicting splits for song {sid}.')
        source_query_totals[source_split] += 1
        source_song_query_totals[sid] += 1
    if source_query_totals != Counter(train=3612, val=766, test=809):
        raise ValueError(f'Embedded source-query totals are invalid: {dict(source_query_totals)}')
    if Counter(source_song_split.values()) != Counter(SOURCE_SONG_COUNTS):
        raise ValueError('Embedded source-query index does not map 300 songs to the verified 210/45/45 splits.')

    found_paths, found_by_song, found_by_split = {}, Counter(), Counter()
    for split, songs in split_ids.items():
        for song in songs:
            row = split_manifest[song]
            song_root = DATASET_SPLIT_DIR / split / song
            if not song_root.is_dir():
                raise FileNotFoundError(song_root)
            for path in song_root.rglob('*.wav'):
                query = SOURCE_QUERY_BY_ID.get(path.stem)
                if query is None:
                    continue
                if path.stem in found_paths:
                    raise ValueError(f'Duplicate source query ID in dataset: {path.stem}')
                if query['song_id'] != row['song_id'] or query['split'] != split:
                    raise ValueError(f'Query is routed to the wrong source song/split: {path}')
                found_paths[path.stem] = path
                found_by_song[query['song_id']] += 1
                found_by_split[split] += 1
                if split in ('train', 'val') and file_sha256(path) != query['humming_sha256']:
                    raise ValueError(f'Source SHA256 mismatch for train/validation query {path.stem}.')
            if row.get('num_hummings') != found_by_song[row['song_id']]:
                raise ValueError(f'Actual query count disagrees with split manifest for {song}.')
            if row.get('source_manifest_num_hummings') != source_song_query_totals[row['song_id']]:
                raise ValueError(f'Source query total disagrees with embedded provenance for {song}.')

    missing_ids = set(SOURCE_QUERY_BY_ID) - set(found_paths)
    if missing_ids and missing_ids != set(KNOWN_MISSING_SOURCE_IDS):
        raise ValueError(f'Unexpected missing source queries: {sorted(missing_ids)[:12]}')
    expected_present = (AVAILABLE_QUERY_COUNTS if missing_ids else {'train': 3612, 'val': 766, 'test': 809})
    if dict(found_by_split) != expected_present:
        raise ValueError(f'Query file counts differ from verified source inventory: {dict(found_by_split)}')
    package_contract = dict(
        final_training_ready=not bool(missing_ids),
        query_files_present_and_sha_verified=dict(found_by_split),
        missing_source_query_files=sorted(missing_ids),
        missing_source_query_counts_by_split=dict(Counter(SOURCE_QUERY_BY_ID[aid]['split'] for aid in missing_ids)),
        source_split_counts_songs=SOURCE_SONG_COUNTS,
        source_query_counts=source_query_totals,
        source_split_mapping_sha256=SOURCE_MAPPING_SHA256,
    )
    if missing_ids:
        if package_contract['missing_source_query_counts_by_split'] != {'test': 7, 'train': 29, 'val': 3}:
            raise ValueError('Known missing query counts differ from the audited partial archive.')
        qbh_stage('Use audited partial source split; train only on available train/validation queries')
        print('[QBH WARNING] Preliminary run: 29 train and 3 validation queries are absent. '
              'No test query is used for training or model selection.', flush=True)
    return package_contract, split_manifest

def split_songs(split):
    directory = DATASET_SPLIT_DIR / split
    if not directory.is_dir():
        raise FileNotFoundError(directory)
    songs = sorted(p.name for p in directory.iterdir() if p.is_dir())
    if not songs:
        raise ValueError(f'Empty split: {split}')
    return songs

REFERENCE_FALLBACKS = []

def scan_queries(split):
    rows = []
    for song in split_songs(split):
        for fragment in sorted((DATASET_SPLIT_DIR / split / song).iterdir()):
            if not fragment.is_dir():
                continue
            files = sorted(p for p in fragment.iterdir() if p.suffix.lower() in AUDIO_EXTENSIONS)
            refs = [p for p in files if p.name == 'reference.wav']
            vocals = [p for p in files if p.name.startswith('reference_vocal')]
            hums = [p for p in files if not p.name.startswith('reference')]
            if hums and not refs:
                full_song = Path(GALLERY_PATHS.get(song, '')).resolve()
                if not full_song.is_file():
                    raise FileNotFoundError(
                        f'No fragment reference and no same-song full recording for {song}: {fragment}'
                    )
                # The package has two audited fragments without reference.wav.
                # Use only that song's own gallery recording as the positive bag.
                refs = [full_song]
                REFERENCE_FALLBACKS.append((split, song, fragment.name))
            rows.extend(dict(song=song, fragment=fragment.name, path=str(h),
                             key=h.relative_to(DATASET_SPLIT_DIR).as_posix(),
                             refs=[str(p) for p in refs], vocals=[str(p) for p in vocals]) for h in hums)
    if not rows:
        raise ValueError(f'No humming queries: {split}')
    if {r['song'] for r in rows} != set(split_songs(split)):
        raise ValueError(f'Songs without humming queries in {split}')
    if any(not row['refs'] for row in rows):
        raise ValueError(f'Positive reference resolution failed in {split}')
    return rows

SPLIT_IDS = {name: split_songs(name) for name in ('train', 'val', 'test')}
if SOLUTION_PHASE == 'train':
    qbh_stage('Repair legacy split and verify locked holdout contract')
    repair_legacy_dataset_split()
    # Folder moves change the split inventory; refresh it before validation.
    SPLIT_IDS = {name: split_songs(name) for name in ('train', 'val', 'test')}
    _package_contract, _source_split_manifest = validate_dataset_package(SPLIT_IDS)
    RUN_CONFIG['dataset_policy'] = ('audited_partial_source_archive'
                                    if not _package_contract.get('final_training_ready', False)
                                    else 'complete_source_archive')
else:
    _package_contract, _source_split_manifest = None, None
for left, right in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    if set(SPLIT_IDS[left]) & set(SPLIT_IDS[right]):
        raise ValueError(f'Song leakage: {left}/{right}')

GALLERY_PATHS = {}
for path in sorted(MUSIC_DB_DIR.iterdir()):
    if path.suffix.lower() not in AUDIO_EXTENSIONS:
        continue
    if path.stem in GALLERY_PATHS:
        raise ValueError(f'Duplicate gallery song: {path.stem}')
    GALLERY_PATHS[path.stem] = str(path)
if set(GALLERY_PATHS) != set().union(*map(set, SPLIT_IDS.values())):
    raise ValueError('Full music gallery must exactly match the train/val/test song IDs.')

TRAIN_ROWS, VAL_ROWS = scan_queries('train'), scan_queries('val')
if REFERENCE_FALLBACKS:
    print(f'[QBH WARNING] {len(REFERENCE_FALLBACKS)} fragments lack reference.wav; '
          'using only the matching song from the full-song gallery as positive audio:')
    for fallback in sorted(REFERENCE_FALLBACKS):
        print('  ', fallback)
if SOLUTION_PHASE == 'train':
    _expected_rows = (AVAILABLE_QUERY_COUNTS if not _package_contract.get('final_training_ready', False)
                      else SOURCE_QUERY_COUNTS)
    if len(TRAIN_ROWS) != _expected_rows['train'] or len(VAL_ROWS) != _expected_rows['val']:
        raise ValueError(
            f'Wrong train/validation query inventory: expected train={_expected_rows["train"]}, '
            f'val={_expected_rows["val"]}; found train={len(TRAIN_ROWS)}, val={len(VAL_ROWS)}. '
            'Check missing or duplicated humming WAVs.'
        )
    _locked_path = DATASET_SPLIT_DIR.parent / 'holdout' / 'locked_final_test_manifest.csv'
    if not _locked_path.is_file():
        raise FileNotFoundError('Missing locked 615-query holdout manifest in dataset package: ' + str(_locked_path))
    if file_sha256(_locked_path) != LOCKED_MANIFEST_SHA256:
        raise ValueError('Locked holdout manifest hash differs from the verified 615-query artifact.')
    _locked_frame = pd.read_csv(_locked_path, dtype=str).fillna('')
    if len(_locked_frame) != 615 or _locked_frame['audio_id'].nunique() != 615:
        raise ValueError(f'Locked final holdout must have exactly 615 unique query IDs; found {len(_locked_frame)} rows.')
    if _locked_frame['song_id'].nunique() != 36 or set(_locked_frame['source_split']) != {'test'}:
        raise ValueError('Locked holdout must contain 36 original-test songs only.')
    _locked_ids = set(_locked_frame['audio_id'])
    _trainval_ids = {Path(row['path']).stem for row in TRAIN_ROWS + VAL_ROWS}
    if _locked_ids & _trainval_ids:
        raise ValueError('Locked test query IDs occur in train/validation; refusing model selection with leakage.')
    _package_root = DATASET_SPLIT_DIR.parent.parent
    for _row in _locked_frame.to_dict('records'):
        _source_row = SOURCE_QUERY_BY_ID.get(_row['audio_id'])
        if not _source_row or _source_row['split'] != 'test' or _source_row['song_id'] != _row['song_id']:
            raise ValueError(f'Locked query disagrees with embedded source provenance: {_row["audio_id"]}')
        _relative = PurePosixPath(str(_row['dataset_relpath']))
        if _relative.parts[:3] != ('datasets', 'dataset_split', 'test'):
            raise ValueError(f'Locked query is not stored under the test split: {_row["dataset_relpath"]}')
        _path = _package_root.joinpath(*_relative.parts)
        if not _path.is_file() or _path.stem != _row['audio_id']:
            raise FileNotFoundError(f'Locked query path is missing or inconsistent: {_path}')
        _song_folder = _relative.parts[3]
        if _source_split_manifest[_song_folder].get('song_id') != _row['song_id']:
            raise ValueError(f'Locked query song ID does not match source mapping: {_row["audio_id"]}')
    if _package_contract.get('final_training_ready', False):
        qbh_success('Verified complete 210/45/45 source split, 3,612/766 train-validation queries and 615 holdout paths; test audio bytes were not read.')
    else:
        qbh_success('Verified 210/45/45 source split, 3,583/763 available train-validation queries and 615 holdout paths; test audio bytes were not read.')
VAL_GALLERY_PATHS = {s: p for s, p in GALLERY_PATHS.items()
                     if s in set(SPLIT_IDS['train'] + SPLIT_IDS['val'])}

def query_records(rows):
    return [(r['key'], r['song'], file_sha256(r['path'])) for r in rows]

def positive_contract_path(path):
    path = Path(path).resolve()
    try:
        return 'split/' + path.relative_to(DATASET_SPLIT_DIR.resolve()).as_posix()
    except ValueError:
        pass
    try:
        return 'gallery/' + path.relative_to(Path(MUSIC_DB_DIR).resolve()).as_posix()
    except ValueError:
        raise ValueError(f'Positive reference is outside the verified split and gallery: {path}')

def data_contract():
    # Training hashes train+val only. Test query audio is never opened here.
    train_q, val_q = query_records(TRAIN_ROWS), query_records(VAL_ROWS)
    if {r[2] for r in train_q} & {r[2] for r in val_q}:
        raise ValueError('Duplicate humming audio across train/val')
    positives = sorted({p for r in TRAIN_ROWS for p in r['refs'] + r['vocals']})
    return dict(split_ids=SPLIT_IDS, train_queries=train_q, val_queries=val_q,
                positive_reference_policy=RUN_CONFIG['positive_selection'],
                full_song_positive_fallbacks=sorted(set(REFERENCE_FALLBACKS)),
                positives=[(positive_contract_path(p), file_sha256(p)) for p in positives],
                val_gallery=[(s, file_sha256(p)) for s, p in VAL_GALLERY_PATHS.items()])

DATA_CONTRACT = json.loads(json.dumps(data_contract()))
DATA_DIGEST = digest(DATA_CONTRACT)
print('Split sizes:', {s: len(ids) for s, ids in SPLIT_IDS.items()})
print('Queries train/val:', len(TRAIN_ROWS), len(VAL_ROWS), '| Validation gallery:', len(VAL_GALLERY_PATHS))

class FinalModel(ConformerByteHumModel):
    def train(self, mode=True):
        super().train(mode)
        if mode:
            encoder = self.feature_extractor._encoder
            self.feature_extractor._global_cmvn.eval()
            # Frozen parameters alone do not freeze dropout or BN running buffers.
            encoder.embed.eval()
            for block in list(encoder.encoders)[:RUN_CONFIG['freeze_blocks']]:
                block.eval()
        return self

class FinalTripletDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
        self.songs = SPLIT_IDS['train']
        self.song_to_idx = {s: i for i, s in enumerate(self.songs)}
        self.humming_samples = [(r['song'], r['fragment'], r['path']) for r in rows]
        self.reference_by_song = {s: sorted({p for r in rows if r['song'] == s for p in r['refs']}) for s in self.songs}
        self.warp = SpectrogramTempoAndMaskDistortion((0.7, 1.3), 8, 4, 0.6)
        self.pitch = CQTPitchShiftAugmentation(12, 0.8)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        a = random.choice(raw_cqts(row['path'])).clone()
        a = self.pitch(self.warp(a))
        use_vocal = row['vocals'] and random.random() < RUN_CONFIG['vocal_ratio']
        pos_path = random.choice(row['vocals'] if use_vocal else row['refs'])
        # Positive identity is always the same source fragment, never another song.
        n_song = random.choice([s for s in self.songs if s != row['song']])
        n = random.choice(raw_cqts(random.choice(self.reference_by_song[n_song]))).clone()
        return dict(anchor=a, positive_path=pos_path, negative=n,
                    song_idx=torch.tensor(self.song_to_idx[row['song']]),
                    negative_song_idx=torch.tensor(self.song_to_idx[n_song]))

@torch.inference_mode()
def encode_cqts(eval_model, cqts):
    gs, ls = [], []
    for start in range(0, len(cqts), EVAL_BATCH_SIZE):
        with torch.autocast('cuda', dtype=torch.float16):
            g, l, _, _ = eval_model(torch.stack(cqts[start:start + EVAL_BATCH_SIZE]).to(device),
                                    compute_lac=False, compute_domain=False)
        gs.append(F.normalize(g.float(), dim=-1).cpu())
        ls.append(F.normalize(l.float(), dim=-1).cpu())
    g, l = torch.cat(gs), torch.cat(ls).reshape(-1, EMBEDDING_DIM)
    if not torch.isfinite(g).all() or not torch.isfinite(l).all():
        raise ValueError('Non-finite embeddings')
    return g, l

@torch.no_grad()
def aligned_positives(eval_model, positive_paths, anchor_global, anchor_local):
    # Multiple-instance positive selection avoids forcing a random unrelated chunk
    # of the 30s source fragment to align with a short humming query.
    if not len(positive_paths) or len(positive_paths) != len(anchor_global) or len(positive_paths) != len(anchor_local):
        raise ValueError('Positive paths and anchor features must have equal nonzero batch size')
    was_training = eval_model.training
    eval_model.eval()
    selected = [None] * len(positive_paths)
    groups = {}
    for index, path in enumerate(positive_paths):
        groups.setdefault(str(path), []).append(index)
    try:
        anchor_global = F.normalize(anchor_global.detach().float(), dim=-1)
        anchor_local = F.normalize(anchor_local.detach().float(), dim=-1)
        # Features are shared only within this call, never across optimizer updates.
        for path, indices in groups.items():
            chunks = raw_cqts(path)
            best_scores = {i: float('-inf') for i in indices}
            best_indices = {i: 0 for i in indices}
            for start in range(0, len(chunks), EVAL_BATCH_SIZE):
                with torch.autocast('cuda', dtype=torch.float16):
                    g, l, _, _ = eval_model(torch.stack(chunks[start:start + EVAL_BATCH_SIZE]).to(device),
                                            compute_lac=False, compute_domain=False)
                g, l = F.normalize(g.float(), dim=-1), F.normalize(l.float(), dim=-1)
                if not torch.isfinite(g).all() or not torch.isfinite(l).all():
                    raise FloatingPointError(f'Non-finite positive selection features: {path}')
                for index in indices:
                    local = torch.einsum('td,nsd->nts', anchor_local[index], l).amax(-1).mean(-1)
                    scores = 0.5 * (g @ anchor_global[index] + local)
                    value, offset = scores.max(dim=0)
                    if float(value) > best_scores[index]:
                        best_scores[index], best_indices[index] = float(value), start + int(offset)
            for index in indices:
                selected[index] = chunks[best_indices[index]].clone()
    finally:
        eval_model.train(was_training)
    return torch.stack(selected).to(device)

def gallery_features(eval_model, paths, checkpoint_hash=None):
    key = digest(dict(checkpoint=checkpoint_hash, records=[(s, file_sha256(p)) for s, p in paths.items()],
                      code=SOURCE_CODE_DIGEST, versions=RUNTIME_VERSIONS, prep=PREPROCESS_CONFIG))
    cache_path = FEATURE_DIR / (key + '.pt')
    if checkpoint_hash is not None and cache_path.is_file():
        gallery = torch.load(cache_path, map_location='cpu', weights_only=True)
    else:
        gallery = {s: encode_cqts(eval_model, raw_cqts(p)) for s, p in tqdm(paths.items(), desc='Full-song features')}
        if checkpoint_hash is not None:
            atomic_torch(cache_path, gallery)
    if list(gallery) != list(paths):
        raise ValueError('Gallery feature cache IDs mismatch')
    for g, l in gallery.values():
        if g.ndim != 2 or l.ndim != 2 or g.shape[1] != 128 or l.shape[1] != 128 or not len(g) or not len(l):
            raise ValueError('Invalid gallery cache shape')
        if not torch.isfinite(g).all() or not torch.isfinite(l).all():
            raise ValueError('Non-finite gallery cache')
    return gallery

def rank_statistics(scores, target):
    values = np.asarray(scores, dtype=np.float64)
    if values.ndim != 1 or not np.isfinite(values).all():
        raise ValueError('All exact-gallery scores must be finite')
    tied = np.isclose(values, values[target], rtol=1e-6, atol=1e-7)
    greater = int(((values > values[target]) & ~tied).sum())
    ties = int(tied.sum())
    rank = 1 + greater + 0.5 * (ties - 1)
    return dict(rank=float(rank), rr=float(1 / rank), ties=ties, greater=greater,
                **{f'recall_at_{k}': float(np.clip((k - greater) / ties, 0, 1)) for k in (1, 5, 10, 20)})

def retrieval_metrics(rows):
    if not rows:
        raise ValueError('Empty evaluation')
    frame = pd.DataFrame(rows)
    result = dict(queries=len(frame), songs=int(frame.song.nunique()))
    for mode in ('local', 'global', 'fused'):
        fields = [mode + '_rr'] + [mode + f'_recall_at_{k}' for k in (1, 5, 10, 20)]
        micro = frame[fields].mean()
        macro = frame.groupby('song')[fields].mean().mean()
        result[mode] = dict(mrr=float(micro.iloc[0]), macro_song_mrr=float(macro.iloc[0]),
                            **{f'recall_at_{k}': float(micro.iloc[i + 1]) for i, k in enumerate((1, 5, 10, 20))})
    return result

@torch.inference_mode()
def evaluate_queries(eval_model, gallery, queries, progress_path=None):
    songs = list(gallery)
    targets = {s: i for i, s in enumerate(songs)}
    completed = json.loads(progress_path.read_text(encoding='utf-8')) if progress_path and progress_path.exists() else []
    keys = [r['key'] for r in queries]
    if [r['key'] for r in completed] != keys[:len(completed)]:
        raise ValueError('Evaluation progress does not match the locked query list')
    for query in tqdm(queries[len(completed):], desc='Exact full-gallery retrieval'):
        torch.cuda.synchronize()
        start = time.perf_counter()
        qg, ql = encode_cqts(eval_model, raw_cqts(query['path']))
        scores = score_gallery(qg, ql, gallery)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        row = dict(key=query['key'], song=query['song'], seconds=elapsed)
        for mode, values in scores.items():
            stats = rank_statistics(values.cpu().numpy(), targets[query['song']])
            row.update({mode + '_' + name: value for name, value in stats.items()})
        completed.append(row)
        if progress_path:
            atomic_json(progress_path, completed)
    if len(completed) != len(queries):
        raise ValueError('Incorrect query denominator')
    return completed



In [ ]:
def validated_best_snapshot(state, run_signature, data_contract):
    if state.get('checkpoint_schema') != 2 or 'best_checkpoint' not in state:
        raise ValueError('Legacy checkpoint: use its original notebook, or start a new run with this code.')
    best = state['best_checkpoint']
    for item in (state, best):
        if item.get('run_signature') != run_signature or item.get('data_contract') != data_contract:
            raise ValueError('Resume contract mismatch')
    metric = state['best_mrr']
    if not math.isfinite(metric) or not 0 <= metric <= 1 or best['best_mrr'] != metric:
        raise ValueError('Invalid or inconsistent best validation metric')
    if not 1 <= best['epoch'] <= state['epoch'] or len(state['history']) != state['epoch']:
        raise ValueError('Invalid best epoch/history in last checkpoint')
    if [row['epoch'] for row in state['history']] != list(range(1, state['epoch'] + 1)):
        raise ValueError('Noncontiguous training history')
    values = [row['val_fused_mrr'] for row in state['history']]
    if not all(math.isfinite(value) and 0 <= value <= 1 for value in values):
        raise ValueError('Non-finite validation history')
    if abs(max(values) - metric) > 1e-8 or values[best['epoch'] - 1] != metric:
        raise ValueError('Best snapshot does not match validation history')
    if best['validation_metrics']['fused']['mrr'] != metric:
        raise ValueError('Best validation report mismatch')
    return best


def export_best_checkpoint(path, snapshot, finished):
    expected = dict(snapshot, finished=bool(finished))
    if Path(path).exists():
        current = torch.load(path, map_location='cpu', weights_only=False)
        metadata = lambda item: {key: value for key, value in item.items() if key != 'model_state_dict'}
        left, right = current.get('model_state_dict', {}), expected['model_state_dict']
        if metadata(current) == metadata(expected) and left.keys() == right.keys() and all(
                left[key].dtype == right[key].dtype and left[key].shape == right[key].shape
                and torch.equal(left[key], right[key]) for key in right):
            return expected  # Preserve checkpoint bytes/hash on completed Run All.
    atomic_torch(path, expected)
    return expected


def commit_training_checkpoint(last_path, best_path, state):
    snapshot = validated_best_snapshot(state, RUN_SIGNATURE, DATA_CONTRACT)
    # LAST is the single atomic commit; BEST/reports can always be rebuilt from it.
    atomic_torch(last_path, state)
    export_best_checkpoint(best_path, snapshot, state['finished'])
    atomic_json(Path(best_path).parent / 'best_validation_metrics.json', snapshot['validation_metrics'])
    atomic_json(Path(last_path).parent / 'history.json', state['history'])


def training_objective(outputs, labels, negative_labels, epoch, memory_bank):
    ga, gp, gn = [F.normalize(out[0].float(), dim=-1) for out in outputs]
    la, lp, ln = [out[1].float() for out in outputs]
    curriculum = mining_hardest_ratio(epoch)
    zero = ga.sum() * 0
    triplet = in_batch_cosine_triplet_loss(ga, gp, gn, labels, negative_labels,
        margin=RUN_CONFIG['margin'], memory_bank=memory_bank, hardest_ratio=curriculum,
        top_k=RUN_CONFIG['mining_top_k']) if RUN_CONFIG['global_triplet'] > 0 else zero
    lat = local_alignment_triplet(la, lp, ln, labels, negative_labels,
        margin=RUN_CONFIG['margin'], hardest_ratio=curriculum,
        top_k=RUN_CONFIG['mining_top_k']) if RUN_CONFIG['lat'] > 0 else zero
    lac = torch.stack([F.cross_entropy(out[2].float(), target) for out, target in zip(outputs, (labels, labels, negative_labels))]).mean() if RUN_CONFIG['lac'] > 0 else zero
    domain = torch.stack([F.binary_cross_entropy_with_logits(out[3].float(), torch.full_like(out[3].float(), target)) for out, target in zip(outputs, (1., 0., 0.))]).mean() if RUN_CONFIG['grl'] > 0 else zero
    prototype = prototype_diversity_loss(model.lac.prototypes.float())
    total = RUN_CONFIG['global_triplet'] * triplet + RUN_CONFIG['lat'] * lat + RUN_CONFIG['lac'] * lac + domain + RUN_CONFIG['prototype_diversity'] * prototype
    return dict(global_triplet=triplet, lat=lat, lac=lac, domain=domain, prototype=prototype, total=total), (ga, gp, gn)


qbh_stage('Find and fingerprint CoverHunter SHS100K Stage-1 checkpoint')
STAGE1_CKPT = unique_path(STAGE1_OVERRIDE, INPUT_ROOT.rglob('coverhunter_shs100k_pretrained.pth'), 'STAGE1_OVERRIDE')
RUN_CONTRACT = dict(config=RUN_CONFIG, preprocessing=PREPROCESS_CONFIG,
                    code=SOURCE_CODE_DIGEST, runtime=RUNTIME_VERSIONS,
                    data=DATA_DIGEST, stage1=file_sha256(STAGE1_CKPT))
qbh_success(f'Found pretrained checkpoint: {STAGE1_CKPT}')
RUN_SIGNATURE = digest(RUN_CONTRACT)
contract_path = TRAIN_OUTPUT / 'run_contract.json'
if contract_path.exists():
    if json.loads(contract_path.read_text(encoding='utf-8')) != RUN_CONTRACT:
        raise ValueError('Run configuration/data/code changed. Use a new OUTPUT_ROOT, never overwrite this run.')
else:
    atomic_json(contract_path, RUN_CONTRACT)

qbh_stage('Initialize model, optimizer, sampler and checkpoint state')
train_dataset = FinalTripletDataset(TRAIN_ROWS)
loader_generator = torch.Generator().manual_seed(42)
sampler = SongBalancedBatchSampler(train_dataset, RUN_CONFIG['songs_per_batch'], RUN_CONFIG['samples_per_song'])
train_loader = DataLoader(train_dataset, batch_sampler=sampler, num_workers=0,
                          pin_memory=True, generator=loader_generator)
model = FinalModel(ckpt_path=str(STAGE1_CKPT), num_classes=len(SPLIT_IDS['train']))
model.freeze_lower_layers(RUN_CONFIG['freeze_blocks'], freeze_frontend=True)
model.to(device)
optimizer = torch.optim.AdamW([
    dict(params=[p for name, p in model.named_parameters() if p.requires_grad and not any(k in name for k in ('lac', 'domain_classifier'))], lr=RUN_CONFIG['lr_backbone']),
    dict(params=[p for name, p in model.named_parameters() if p.requires_grad and any(k in name for k in ('lac', 'domain_classifier'))], lr=RUN_CONFIG['lr_heads'])
], weight_decay=RUN_CONFIG['weight_decay'])
updates_per_epoch = math.ceil(len(train_loader) / RUN_CONFIG['accumulation'])
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer,
    max_lr=[RUN_CONFIG['lr_backbone'], RUN_CONFIG['lr_heads']],
    total_steps=updates_per_epoch * RUN_CONFIG['epochs'], pct_start=0.1)
scaler = torch.amp.GradScaler('cuda', init_scale=1024)
memory = CrossBatchMemoryBank(RUN_CONFIG['memory_capacity'])
best_path, last_path = TRAIN_OUTPUT / BEST_NAME, TRAIN_OUTPUT / LAST_NAME
history, best_mrr, patience, start_epoch, finished = [], -1.0, 0, 1, False
best_snapshot = None
qbh_success(f'Model initialized with {len(SPLIT_IDS["train"])} train songs and {len(TRAIN_ROWS)} train queries')

if RESUME:
    qbh_stage('Restore a compatible LAST checkpoint or start from Stage 1')
    source = last_path if last_path.exists() else (Path(RESUME_LAST_OVERRIDE) if RESUME_LAST_OVERRIDE else None)
    if source is not None:
        # RNG tensors must remain on CPU even when the model/optimizer live on CUDA.
        state = torch.load(source, map_location='cpu', weights_only=False)
        best_snapshot = validated_best_snapshot(state, RUN_SIGNATURE, DATA_CONTRACT)
        model.load_state_dict(state['model_state_dict'], strict=True)
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        history, best_mrr, patience = state['history'], state['best_mrr'], state['patience']
        start_epoch, finished = state['epoch'] + 1, state['finished']
        random.setstate(state['rng_python'])
        np.random.set_state(state['rng_numpy'])
        torch.set_rng_state(state['rng_torch'])
        torch.cuda.set_rng_state_all(state['rng_cuda'])
        loader_generator.set_state(state['rng_loader'])
        commit_training_checkpoint(last_path, best_path, state)
        del state
        print('Resuming epoch:', start_epoch, '| Best fused validation MRR:', best_mrr)
        qbh_success(f'Restored LAST checkpoint; next epoch={start_epoch}')
    elif best_path.exists() or RESUME_BEST_OVERRIDE is not None:
        raise FileNotFoundError('BEST alone cannot resume optimizer/RNG. Attach the matching LAST or use a fresh OUTPUT_ROOT.')
elif last_path.exists() or best_path.exists():
    raise FileExistsError('Existing run: enable RESUME or choose a new OUTPUT_ROOT.')
qbh_success('Resume contract is valid; training can proceed')

def model_cpu_state():
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def base_checkpoint(epoch, complete=False):
    return dict(model_state_dict=model_cpu_state(), run_signature=RUN_SIGNATURE,
                run_contract=RUN_CONTRACT, data_contract=DATA_CONTRACT, epoch=epoch,
                best_mrr=best_mrr, finished=complete, train_songs=SPLIT_IDS['train'], checkpoint_schema=2)

def gpu_preflight():
    saved_rng = (random.getstate(), np.random.get_state(), torch.get_rng_state(),
                 torch.cuda.get_rng_state_all(), loader_generator.get_state())
    saved_weights, was_training = model_cpu_state(), model.training
    saved_lambdas = [(module, module.lambda_val) for module in model.modules() if hasattr(module, 'lambda_val')]
    probe_optimizer = torch.optim.AdamW([dict(group, params=list(group['params'])) for group in optimizer.param_groups], **optimizer.defaults)
    probe_scaler = torch.amp.GradScaler('cuda', init_scale=1.0)
    probe_memory = CrossBatchMemoryBank(RUN_CONFIG['memory_capacity'])
    model.train()
    try:
        probe_optimizer.zero_grad(set_to_none=True)
        batches = iter(train_loader)
        group_size = min(RUN_CONFIG['accumulation'], len(train_loader))
        if group_size < 1:
            raise ValueError('Empty training loader')
        for step in range(group_size):
            batch = next(batches)
            progress = ((start_epoch - 1) * len(train_loader) + step) / (RUN_CONFIG['epochs'] * len(train_loader))
            model.set_lambda(RUN_CONFIG['grl'] * (2 / (1 + math.exp(-10 * progress)) - 1))
            a, n = [batch[key].to(device) for key in ('anchor', 'negative')]
            labels, negative_labels = batch['song_idx'].to(device), batch['negative_song_idx'].to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                oa = model(a, compute_lac=RUN_CONFIG['lac'] > 0, compute_domain=RUN_CONFIG['grl'] > 0)
            p = aligned_positives(model, batch['positive_path'], oa[0], oa[1])
            with torch.autocast('cuda', dtype=torch.float16):
                outputs = [oa] + [model(x, compute_lac=RUN_CONFIG['lac'] > 0, compute_domain=RUN_CONFIG['grl'] > 0) for x in (p, n)]
            expected = (RUN_CONFIG['songs_per_batch'] * RUN_CONFIG['samples_per_song'], 128)
            if oa[0].shape != expected or oa[1].shape != (expected[0], 20, 128):
                raise ValueError('Model output shape mismatch')
            terms, globals_ = training_objective(outputs, labels, negative_labels, start_epoch, probe_memory)
            if not torch.isfinite(terms['total']):
                raise FloatingPointError('GPU preflight loss is not finite')
            probe_scaler.scale(terms['total'] / group_size).backward()
            probe_memory.enqueue(torch.cat(globals_), torch.cat((labels, labels, negative_labels)))
            del terms, globals_, outputs, oa, a, p, n
        probe_scaler.unscale_(probe_optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), RUN_CONFIG['clip'])
        if not torch.isfinite(norm):
            raise FloatingPointError('GPU preflight gradient is not finite')
        previous_scale = probe_scaler.get_scale()
        probe_scaler.step(probe_optimizer)
        probe_scaler.update()
        if probe_scaler.get_scale() < previous_scale:
            raise FloatingPointError('GPU preflight optimizer update was skipped')
        if any(not torch.isfinite(p).all() for p in model.parameters()):
            raise FloatingPointError('GPU preflight optimizer produced non-finite parameters')
        if not any(p.requires_grad and not torch.equal(p.detach().cpu(), saved_weights[name]) for name, p in model.named_parameters()):
            raise RuntimeError('GPU preflight optimizer did not change any trainable parameter')
        model.eval()
        g, l = encode_cqts(model, raw_cqts(TRAIN_ROWS[0]['path']))
        scores = score_gallery(g, l, {'self': (g, l), 'other': (F.normalize(torch.randn_like(g), dim=-1), F.normalize(torch.randn_like(l), dim=-1))})
        if float(scores['fused'][0]) < 0.999:
            raise ValueError('GPU exact scoring identity check failed')
        print('GPU preflight passed: train-mode AMP accumulation, all configured losses, AdamW update and exact scoring. Probe state is rolled back.')
    finally:
        model.load_state_dict(saved_weights, strict=True)
        for module, value in saved_lambdas:
            module.lambda_val = value
        optimizer.zero_grad(set_to_none=True)
        random.setstate(saved_rng[0])
        np.random.set_state(saved_rng[1])
        torch.set_rng_state(saved_rng[2])
        torch.cuda.set_rng_state_all(saved_rng[3])
        loader_generator.set_state(saved_rng[4])
        model.train(was_training)
        del probe_optimizer, probe_scaler, probe_memory, saved_weights
        gc.collect()
        torch.cuda.empty_cache()

if not finished:
    qbh_stage('GPU preflight: AMP, configured losses, optimizer update and scorer identity')
    gpu_preflight()
    qbh_success('GPU preflight passed; probe weights and RNG were rolled back')
    qbh_stage('Prewarm and validate all train/validation audio CQT features')
    prewarm_paths = sorted({r['path'] for r in TRAIN_ROWS + VAL_ROWS}
                           | {p for r in TRAIN_ROWS for p in r['refs'] + r['vocals']}
                           | set(VAL_GALLERY_PATHS.values()))
    for path in tqdm(prewarm_paths, desc='Validate audio and cache complete CQT'):
        raw_cqts(path)
    qbh_success(f'Audio/CQT preflight passed for {len(prewarm_paths)} unique train/validation/reference/gallery files')

if not finished:
    for epoch in range(start_epoch, RUN_CONFIG['epochs'] + 1):
        qbh_stage(f'Train epoch {epoch}/{RUN_CONFIG["epochs"]}')
        model.train()
        memory.clear()
        optimizer.zero_grad(set_to_none=True)
        losses = dict(global_triplet=0., lat=0., lac=0., domain=0., prototype=0., total=0.)
        skipped_updates = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}')):
            progress = ((epoch - 1) * len(train_loader) + step) / (RUN_CONFIG['epochs'] * len(train_loader))
            model.set_lambda(RUN_CONFIG['grl'] * (2 / (1 + math.exp(-10 * progress)) - 1))
            a, n = [batch[key].to(device, non_blocking=True) for key in ('anchor', 'negative')]
            labels, negative_labels = batch['song_idx'].to(device), batch['negative_song_idx'].to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                oa = model(a, compute_lac=RUN_CONFIG['lac'] > 0, compute_domain=RUN_CONFIG['grl'] > 0)
            p = aligned_positives(model, batch['positive_path'], oa[0], oa[1])
            with torch.autocast('cuda', dtype=torch.float16):
                outputs = [oa] + [model(x, compute_lac=RUN_CONFIG['lac'] > 0, compute_domain=RUN_CONFIG['grl'] > 0) for x in (p, n)]
            terms, globals_ = training_objective(outputs, labels, negative_labels, epoch, memory)
            total = terms['total']
            if not bool(torch.isfinite(total)):
                raise FloatingPointError(f'Non-finite loss at epoch {epoch}, batch {step}')
            # The last accumulation group may have fewer microbatches.
            group_start = (step // RUN_CONFIG['accumulation']) * RUN_CONFIG['accumulation']
            group_size = min(RUN_CONFIG['accumulation'], len(train_loader) - group_start)
            scaler.scale(total / group_size).backward()
            memory.enqueue(torch.cat(globals_), torch.cat((labels, labels, negative_labels)))
            for key, value in terms.items():
                losses[key] += float(value.detach())
            if (step + 1) % RUN_CONFIG['accumulation'] == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), RUN_CONFIG['clip'])
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= previous_scale:
                    scheduler.step()
                else:
                    skipped_updates += 1
                optimizer.zero_grad(set_to_none=True)
            del outputs, oa, globals_, terms, total, a, p, n, value
        if skipped_updates == updates_per_epoch:
            raise FloatingPointError('All optimizer updates were skipped by AMP; do not accept this epoch.')
        qbh_stage(f'Validate epoch {epoch} on the full train+validation gallery')
        model.eval()
        gallery = gallery_features(model, VAL_GALLERY_PATHS)
        val_rows = evaluate_queries(model, gallery, VAL_ROWS)
        metrics = retrieval_metrics(val_rows)
        del gallery
        val_mrr = metrics['fused']['mrr']
        history.append(dict(epoch=epoch, **{key: value / len(train_loader) for key, value in losses.items()},
                            val_fused_mrr=val_mrr, val_macro_mrr=metrics['fused']['macro_song_mrr'],
                            val_recall_at_1=metrics['fused']['recall_at_1'], skipped_updates=skipped_updates))
        if val_mrr > best_mrr + 1e-8:
            best_mrr, patience = val_mrr, 0
            best_snapshot = dict(base_checkpoint(epoch), validation_metrics=metrics)
        else:
            patience += 1
        finished = patience >= RUN_CONFIG['patience'] or epoch == RUN_CONFIG['epochs']
        state = base_checkpoint(epoch, finished)
        state.update(optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(), scaler=scaler.state_dict(),
                     history=history, patience=patience, rng_python=random.getstate(), rng_numpy=np.random.get_state(),
                     rng_torch=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state_all(), rng_loader=loader_generator.get_state(),
                     best_checkpoint=best_snapshot)
        commit_training_checkpoint(last_path, best_path, state)
        del state
        print(f'Epoch {epoch}: fused val MRR={val_mrr:.4f}, best={best_mrr:.4f}, patience={patience}')
        qbh_success(f'Epoch {epoch} finished and LAST/BEST checkpoints were committed')
        if finished:
            break

if not finished or not best_path.exists():
    raise RuntimeError('Training is not complete. Resume it before running final evaluation.')
qbh_stage('Finalize checkpoint, receipt and training summary')
best = export_best_checkpoint(best_path, best_snapshot, finished=True)
receipt = dict(run_signature=RUN_SIGNATURE, checkpoint_sha256=file_sha256(best_path),
               best_epoch=best['epoch'], best_validation_fused_mrr=best['best_mrr'],
               completed_epochs=len(history), test_evaluated=False,
               dataset_status='partial_audited_source_archive' if not _package_contract.get('final_training_ready', False)
                              else 'complete_source_archive',
               train_queries=len(TRAIN_ROWS), validation_queries=len(VAL_ROWS),
               known_missing_train_queries=29 if not _package_contract.get('final_training_ready', False) else 0,
               known_missing_validation_queries=3 if not _package_contract.get('final_training_ready', False) else 0)
atomic_json(TRAIN_OUTPUT / 'final_manifest.json', receipt)
model.load_state_dict(best['model_state_dict'])
model.eval()
print('Training complete. Best checkpoint:', best_path)
display(pd.DataFrame([receipt]))
qbh_success(f'Training complete: best epoch={best["epoch"]}, validation fused MRR={best["best_mrr"]:.6f}, checkpoint={best_path}')



In [ ]:
frame = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
frame.set_index('epoch')[['global_triplet', 'lat']].plot(ax=axes[0], title='Training retrieval losses')
frame.set_index('epoch')[['lac', 'domain', 'prototype']].plot(ax=axes[1], title='Training auxiliary losses')
frame.set_index('epoch')[['val_fused_mrr', 'val_macro_mrr']].plot(ax=axes[2], title='Validation exact fused retrieval')
fig.tight_layout()
fig.savefig(TRAIN_OUTPUT / 'training_curves.png', dpi=160)
plt.show()
print('Attach this training output to the separate final evaluation notebook. No test queries were evaluated here.')



## Download training outputs


In [ ]:
def package_training_outputs(training_dir, working_dir):
    from pathlib import Path
    import hashlib
    import json
    import os
    import zipfile

    training_dir, working_dir = Path(training_dir), Path(working_dir)
    required = ['qbh_final_v5_best.pth', 'qbh_final_v5_last.pth', 'final_manifest.json']
    for name in required:
        if not (training_dir / name).is_file():
            raise FileNotFoundError('Training did not finish or output is missing: ' + name)
    receipt = json.loads((training_dir / 'final_manifest.json').read_text(encoding='utf-8'))
    h = hashlib.sha256()
    with (training_dir / 'qbh_final_v5_best.pth').open('rb') as handle:
        for part in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(part)
    if receipt.get('checkpoint_sha256') != h.hexdigest():
        raise ValueError('Best checkpoint does not match the completed-training manifest.')
    names = required + ['run_contract.json', 'history.json', 'best_validation_metrics.json',
                        'training_curves.png', 'run_diagnostics.jsonl']
    archive = working_dir / 'qbh_final_v5_train_outputs.zip'
    temp = archive.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
        for name in names:
            source = training_dir / name
            if source.is_file():
                zipped.write(source, arcname='train/' + name)
    os.replace(temp, archive)
    return archive

qbh_stage('Validate and package final training outputs')
training_archive = package_training_outputs(TRAIN_OUTPUT, Path(OUTPUT_ROOT).parent)
qbh_success('Training outputs validated and packaged', final=True)
print('Training finished. Download checkpoints from Kaggle Output:')
print(TRAIN_OUTPUT / BEST_NAME)
print(TRAIN_OUTPUT / LAST_NAME)
print('Checkpoint-only archive (dataset and CQT cache are excluded):', training_archive)
